# GeoNUSAF — TC-SegFormer **legacy-recipe** rebuild

**The architecture of `5__tc-segformer.ipynb`. The training logic, scoring choices and structure of
`segformer-b0-legacy-parity.ipynb`.**

6 classes with `ignore_index=255` by default, `KFold(5, shuffle=True, random_state=42)` over image
indices, inverse-frequency class weights over the whole dataset, `AdamW(lr=3e-5)` flat with no
scheduler / no clipping / no EMA, 300 epochs, patience 25, batch 16, AMP, the
`HorizontalFlip + VerticalFlip + RandomRotate90 + Resize` albumentations pipeline, a **clean**
validation transform, and `val_miou_core6` as the selection metric.

Everything in the architecture column is TC-SegFormer unchanged: the residual H/2 detail path, CSA
reliability reweighting, soft-clDice on Road/River, and the class-presence-balanced sampler.

## What changed relative to each parent notebook

| | `5__tc-segformer.ipynb` | here |
|---|---|---|
| classes | 6 + `ignore_index=255` | **same** (`IGNORE_UNLABELLED=True`); `False` gives the legacy 7-class scheme |
| split | 3-fold sequence-block over 408 base tiles | **`KFold(5)` over 1224 image indices** (+ block/random controls) |
| class weights | effective-number, cap 10, **train split only** | **legacy `1/n`, uncapped, whole dataset** |
| optimizer | 3-tier LR 6e-6 / 6e-5 / 6e-4, cosine + 150-step warmup | **flat 3e-5**, no schedule (new modules: see §12) |
| clipping / EMA | `clip_grad_norm_(1.0)`, EMA 0.999, val on EMA | **none / none** |
| epochs / patience | 120 / 25 with `min_delta` 0.002 | **300 / 25**, no `min_delta` |
| augmentation | rot90 + brightness/contrast, no flips | **hflip + vflip + rot90** (albumentations) |
| selection metric | pooled 6-class mIoU | **`val_miou_core6`** — the same quantity, now a named switch |
| determinism | `use_deterministic_algorithms(True)` | **off** — ~6x wall-clock otherwise, and 300 epochs would not fit |
| `persistent_workers` | `True` | **`False`** — see the frozen-augmentation note below |

| | `segformer-b0-legacy-parity.ipynb` | here |
|---|---|---|
| model | `SegformerForSemanticSegmentation` | **TC-SegFormer** (residual H/2 detail path) |
| loss | plain weighted CE | **CE x CSA + Lovasz + soft-clDice**, per the `RECIPE` ladder in §2 |
| sampler | `shuffle=True` | **class-presence-balanced** (`RECIPE`-dependent) |
| cache | images + labels | **+ a CSA reliability-weight map** (§6) |
| metrics | pooled + core6 | **+ a CSA core-region pass** (§13, §19) |

Everything else — split, weights, loss family, base LR, schedule, clipping, EMA, epochs, patience,
batch, workers, augmentation, normalisation, scoring, persistence, section structure — follows the
B0 notebook line for line, so the two are directly pairable.

## Two things to read before you trust the number this produces

**1. Under `legacy_kfold`, ~96% of validation images have an exact mirror of themselves in
training.** `KFold` splits the 1224 *files*, and each base tile exists three times on disk as
`imageN.png` / `imageNhf.png` / `imageNvf.png`. §8 measures this for your fold and prints it. The
absolute mIoU from a `legacy_kfold` run is therefore not a segmentation result and must not go in
the paper in any column.

What it *is* good for: TC and B0 leak identically under the same fold, so the **TC minus B0 delta
is a valid paired comparison**. §21 computes exactly that. Run `SPLIT_MODE = "block"` as well and
you get the same delta under the reported protocol, which is the one that can be published.

**2. The old TC-SegFormer runs almost certainly had frozen augmentation.** `5__tc-segformer.ipynb`
builds its train loader with `persistent_workers=True` and calls `train_ds.set_epoch(epoch)` in the
**parent** process. Persistent workers hold their own copy of the dataset and never see that
assignment, so the augmentation seed `(epoch * 100003 + j)` stayed pinned at `epoch = 0` for the
whole run: every tile got the same rot90 angle and the same brightness jitter in epoch 1 and in
epoch 119. This is the same defect already logged against U-Net, SegFormer-B0 and SegNeXt in the
deviation ledger, and TC belongs on that list. It is fixed here (§3, §10), and it is the single most
plausible source of a genuine improvement in this rebuild.

## Prerequisites
- **Internet ON**. Kaggle Secrets: `HF_TOKEN` (write) + `WANDB_API_KEY`. GPU: T4 x1.
- One fold per session. Set `SPLIT_MODE`, `FOLD` and `RECIPE` in §2, run top to bottom.
- Budget 5-8 h per fold at 300 epochs; resume from `last.pt` is automatic.

## 1. Environment

`albumentations` is what the legacy script augments with, so it is imported rather than
re-implemented — an approximation of `A.RandomRotate90` is exactly the kind of silent divergence
this notebook exists to avoid. `scipy.ndimage` is needed for the CSA connected-component means,
`skimage` for true skeletonisation in the final clDice pass, and `ptflops` for the GFLOPs column.

`ptflops` rather than `fvcore` is deliberate. The B0 notebook counts with `ptflops`, and a TC-vs-B0
FLOP comparison across two different counters is meaningless — TC strictly contains B0, so any table
where TC reads *lower* is a counter artefact rather than a measurement. One tool, both rows.

In [ ]:
!pip install -q ptflops==0.7.3 2>&1 | tail -2

import os, re, gc, json, time, math, random, warnings, itertools, copy, shutil
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

warnings.filterwarnings("ignore")
cv2.setNumThreads(0)   # critical: cv2 threads fight DataLoader workers on Kaggle's 4 vCPUs

import albumentations as A
from albumentations.pytorch import ToTensorV2
import transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| albumentations", A.__version__, "| cuda", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

## 2. Configuration

**Set `SPLIT_MODE`, `FOLD` and `RECIPE` here, then run the whole notebook.** One fold per session.

### `SPLIT_MODE`
- `"legacy_kfold"` — `KFold(5, shuffle=True, random_state=42)` over the **1224 image indices**.
  What the legacy script does, and what lets flip variants of one tile straddle the split.
- `"random"` — random split over the **408 base tiles**, variants grouped. Removes duplicate
  leakage, keeps random geography. Use `N_FOLDS = 3`.
- `"block"` — contiguous export-order blocks over base tiles. The reported protocol. `N_FOLDS = 3`.

### `IGNORE_UNLABELLED`
`True` (default) trains and scores **6 classes** with black pixels at `ignore_index=255`, matching
the rest of your benchmark and the B0 notebook's default. `False` restores the legacy 7-class scheme
where unlabelled is a trained class. `N_CLASSES` and `CLASS_NAMES` are **derived** from it so the
two can never disagree, and the scheme appears in the run name as `c6` / `c7`, because a 6-way head
cannot load a 7-way `last.pt`.

This matters more for TC than it does for B0. Under the 7-class scheme the unlabelled blobs get a
CSA band, a Lovasz term and a slot in the balanced sampler's rarity score — three TC components all
spending capacity on annotation mistakes, and the rarest class draws the largest inverse-frequency
weight on top of that. Under `c6` they are simply absent from every one of them.

### `RECIPE` — the three-rung ladder
The legacy recipe and TC-SegFormer disagree about the loss, so which components are on is a
deliberate choice rather than something left implicit. One word changes it:

| `RECIPE` | detail path | CSA | sampler | Lovasz | clDice | what it answers |
|---|---|---|---|---|---|---|
| `"legacy_pure"` | on | off | off | off | off | strict parity — the only difference from the B0 run is the architecture |
| `"tc_lite"` | on | on | on | off | off | adds the two components that introduce no new loss term |
| `"tc_full"` | on | on | on | on | on | TC-SegFormer's full objective under the legacy optimiser |

Default is `"tc_full"`: the literal reading of "TC-SegFormer trained with the B0 recipe".
`"legacy_pure"` is the run to do second — it is the clean A/B against the B0 number, and the gap
between the two says whether TC's loss components still earn their place once the split, the
schedule and the metric all change underneath them.

### The two fields that are not in either parent

`LR_MODE` resolves a genuine conflict. The legacy recipe is `AdamW(model.parameters(), lr=3e-5)` —
one group, everything flat. But `proj` / `detail` / `fuse` / `refine` are randomly initialised, and
`refine` starts at `std = 1e-4` by design so that step 0 is vanilla SegFormer. At a flat 3e-5 those
modules move very slowly and TC risks collapsing into B0 with extra FLOPs. `"new_boosted"` holds the
encoder and pretrained decode head at exactly 3e-5 and gives only the new modules 10x — the same
new:pretrained ratio the original TC notebook used (6e-4 vs 6e-5). `"legacy_flat"` is the stricter
option and is worth one run if the gate ends up near its init value.

`CSA_DMAX_M = 14.07` at `TILE_METRES = 703.5` is **numerically the same band** the original TC runs
used. That notebook wrote `6.0 m` at `TILE_METRES = 300.0`, giving `6.0 * 2345/300 = 46.90 px`; here
`14.07 * 2345/703.5 = 46.90 px`. Identical pixels, correct units — 300 m was the superseded tile-size
assumption and the recovered scale is 0.300 m/px over a 703.5 m tile. Do not "fix" this to 6.0 or you
silently change the band by a factor of 2.345 and break comparability with every TC run on the Hub.

In [ ]:
@dataclass
class CFG:
    # ---- run selection (change these per session) ----
    SPLIT_MODE: str = "legacy_kfold"   # "legacy_kfold" | "random" | "block"
    FOLD: int = 0                      # 0 .. N_FOLDS-1
    RECIPE: str = "tc_full"            # "legacy_pure" | "tc_lite" | "tc_full"

    # ---- data ----
    DATA_ROOT: str = "/kaggle/input/datasets/sugam24/satellite-datasets/satellite_dataset"
    CACHE_DIR: str = "/kaggle/working/cache512"
    PREBUILT_CACHE: Optional[str] = None   # saved image/label cache -> skips the repack
    CSA_CACHE_DIR: str = "/kaggle/working/csa512"   # ALWAYS writable; /kaggle/input is read-only
    PREBUILT_CSA_CACHE: Optional[str] = None
    IMG_SIZE: int = 512
    IGNORE_UNLABELLED: bool = True     # True = 6 classes, black pixels excluded from loss+metrics
                                       # False = the legacy 7-class scheme (unlabelled IS a class)
    IGNORE_INDEX: int = 255
    N_CLASSES: int = 6                 # DERIVED below from IGNORE_UNLABELLED -- do not set by hand
    CLASS_NAMES: tuple = ()            # DERIVED below
    PRELOAD_RAM: bool = True

    # ---- split (legacy) ----
    N_FOLDS: int = 5                   # legacy KFold(n_splits=5); use 3 for block/random
    KFOLD_SEED: int = 42               # legacy random_state=42
    BLOCK_BUFFER: int = 2              # only used when SPLIT_MODE == "block"

    # ---- model: TC-SegFormer ----
    HF_MODEL_ID: str = "nvidia/segformer-b0-finetuned-ade-512-512"
    ARCH_VERSION: str = "v2-residual"  # bump on any module-layout change; guards HF resume
    RESUME: bool = True
    USE_DETAIL: bool = True
    DETAIL_CH: tuple = (8, 16, 32)
    FUSE_CH: int = 64
    DETAIL_GATE_INIT: float = 0.1
    REFINE_INIT_STD: float = 1e-4      # so step 0 == vanilla SegFormer-B0
    DROP_PATH: Optional[float] = None  # legacy passed no overrides -> checkpoint defaults

    # ---- optimization: every value below is literal from the legacy script ----
    BATCH_SIZE: int = 16
    LR: float = 3e-5                   # flat, encoder and pretrained head
    LR_MODE: str = "new_boosted"       # "new_boosted" | "legacy_flat"
    LR_NEW_MULT: float = 10.0          # only used by "new_boosted"
    WEIGHT_DECAY: float = 0.01         # torch AdamW default; legacy passed none
    MAX_EPOCHS: int = 300
    PATIENCE: int = 25
    AMP: bool = True
    DROP_LAST: bool = False            # legacy DataLoader left this at its default

    # ---- deliberately absent in the legacy script ----
    SCHEDULER: Optional[str] = None    # no warmup, no cosine: LR is constant
    CLIP_NORM: Optional[float] = None  # no clip_grad_norm_ (norm still logged)
    LABEL_SMOOTHING: float = 0.0
    USE_EMA: bool = False
    EMA_DECAY: float = 0.999

    # ---- class weights ----
    CLASS_WEIGHT_MODE: str = "legacy_inv_freq"   # "legacy_inv_freq" | "effective_number" | "none"
    WEIGHT_SOURCE: str = "all"                   # legacy counted over train+val together
    WEIGHT_CAP: Optional[float] = None           # legacy applied no cap
    EN_BETA_CAP: float = 10.0                    # only used by "effective_number"

    # ---- TC component C: loss extras and sampler (gated by RECIPE) ----
    LAMBDA_LOVASZ: float = 0.75
    LOVASZ_WARMUP_EPOCHS: int = 10
    CLDICE_MU: float = 0.30
    CLDICE_START_EPOCH: int = 20
    CLDICE_CLASSES: tuple = (1, 2)     # Road, River
    CLDICE_ITERS: int = 5
    CLDICE_RES: int = 256
    SAMPLER_CAP: float = 8.0

    # ---- TC component D: coarse-supervision-aware reweighting (gated by RECIPE) ----
    CSA_TAU: tuple = (0.60, 0.35, 0.35, 0.60, 0.60, 0.60, 0.60)   # 7th used only in the c7 scheme
    CSA_DMAX_M: float = 14.07          # == the original 46.90 px band, in CORRECTED metres
    TILE_METRES: float = 703.5         # recovered tile extent (was 300.0 under the old assumption)
    CSA_WMIN: float = 0.25
    CSA_LOVASZ_THRESH: float = 0.60    # below this, pixels are IGNORED by Lovasz (mask, not weight)
    CORE_THRESH: float = 0.999         # w >= this defines the "core" evaluation region

    # ---- augmentation (legacy albumentations pipeline) ----
    AUG_HFLIP_P: float = 0.5
    AUG_VFLIP_P: float = 0.5
    AUG_ROT90_P: float = 0.5
    VAL_USES_TRAIN_TF: bool = False    # legacy was True; see the note in section 10
    PREPROCESS: str = "processor"      # "processor" (legacy) | "manual" (verified equivalent)

    # ---- which series selects best.pt ----
    SELECTION_METRIC: str = "val_miou_core6"   # "val_miou_core6" | "val_miou_global" | "val_miou_legacy"

    # ---- runtime ----
    SEED: int = 42
    NUM_WORKERS: int = 2               # legacy value
    PERSISTENT_WORKERS: bool = False   # legacy value, and the fix for the frozen-aug bug

    # ---- persistence ----
    HF_USER: str = "sugam24"                            # TC checkpoints live here
    HF_PREFIX: str = "geonusaf-tcsegformer-legacy"
    B0_HF_USER: str = "Pranilllllll"                    # the B0 twin, for section 21
    B0_HF_PREFIX: str = "geonusaf-segformer-b0-legacy"
    HF_PUSH: bool = True
    PUSH_LAST_EVERY: int = 10
    WANDB_PROJECT: str = "geonusaf-baselines-legacy"
    OUT_DIR: str = "/kaggle/working/out"

    ALL_NAMES: tuple = ("Residential", "Road", "River", "Forest",
                        "UnusedLand", "Agricultural", "Unlabelled")

cfg = CFG()
os.makedirs(cfg.OUT_DIR, exist_ok=True)
assert cfg.SPLIT_MODE in ("legacy_kfold", "random", "block")
assert cfg.LR_MODE in ("new_boosted", "legacy_flat")

# Derived, so the two schemes can never disagree with each other:
cfg.N_CLASSES = 6 if cfg.IGNORE_UNLABELLED else 7
cfg.CLASS_NAMES = cfg.ALL_NAMES[:cfg.N_CLASSES]
# Label written for a black pixel: the ignore sentinel, or class 6 in the legacy scheme.
VOID_LABEL = cfg.IGNORE_INDEX if cfg.IGNORE_UNLABELLED else 6
CORE_CLASSES = list(range(6))          # the six real land-use classes, always
assert cfg.SELECTION_METRIC in ("val_miou_core6", "val_miou_global", "val_miou_legacy")
assert len(cfg.CSA_TAU) >= cfg.N_CLASSES, "CSA_TAU needs one entry per scored class"

# ---- the RECIPE ladder: one string derives every component toggle ----
RECIPES = {
    "legacy_pure": dict(CSA_ENABLE=False, BALANCED_SAMPLER=False, LOVASZ_ENABLE=False, CLDICE_ENABLE=False),
    "tc_lite":     dict(CSA_ENABLE=True,  BALANCED_SAMPLER=True,  LOVASZ_ENABLE=False, CLDICE_ENABLE=False),
    "tc_full":     dict(CSA_ENABLE=True,  BALANCED_SAMPLER=True,  LOVASZ_ENABLE=True,  CLDICE_ENABLE=True),
}
assert cfg.RECIPE in RECIPES, f"RECIPE must be one of {list(RECIPES)}"
R = RECIPES[cfg.RECIPE]
CSA_ENABLE       = R["CSA_ENABLE"]
BALANCED_SAMPLER = R["BALANCED_SAMPLER"]
LOVASZ_ENABLE    = R["LOVASZ_ENABLE"]
CLDICE_ENABLE    = R["CLDICE_ENABLE"]

# The class scheme and the recipe both go in the run name: a 6-class head cannot load a 7-class
# checkpoint, and legacy_pure is a different experiment from tc_full.
SCHEME_TAG = "c6" if cfg.IGNORE_UNLABELLED else "c7"
RUN_NAME = f"{cfg.HF_PREFIX}-{SCHEME_TAG}-{cfg.RECIPE}-{cfg.SPLIT_MODE}-fold{cfg.FOLD}"
HF_REPO  = f"{cfg.HF_USER}/{RUN_NAME}"
print("RUN:", RUN_NAME, "| HF repo:", HF_REPO)
print(f"recipe {cfg.RECIPE}: detail={cfg.USE_DETAIL} csa={CSA_ENABLE} sampler={BALANCED_SAMPLER} "
      f"lovasz={LOVASZ_ENABLE} cldice={CLDICE_ENABLE}")
print(f"legacy recipe: lr={cfg.LR} flat ({cfg.LR_MODE}) | epochs={cfg.MAX_EPOCHS} | "
      f"patience={cfg.PATIENCE} | weights={cfg.CLASS_WEIGHT_MODE} | no sched/clip/EMA")
print(f"scoring: {cfg.N_CLASSES} classes "
      f"({'unlabelled IGNORED' if cfg.IGNORE_UNLABELLED else 'unlabelled is class 6'}) | "
      f"val transform: {'train (legacy)' if cfg.VAL_USES_TRAIN_TF else 'clean'} | "
      f"select on {cfg.SELECTION_METRIC}")

## 3. Determinism

The legacy script seeds nothing, so it is not reproducible — repeating it would give a different
number each time. Seeding here is a strict improvement and does not touch the update rule.

`use_deterministic_algorithms` is left **off**, reversing the setting in `5__tc-segformer.ipynb`. It
forces a sort-based backward kernel for the bilinear upsample that costs roughly 6x wall-clock on
this workload, and at 300 epochs that is the difference between a fold fitting in a Kaggle session
and not. Reproducibility is "same seed, same machine, same library versions" rather than bit-exact —
state it that way in the paper. The side benefit is that wall-clock and FPS from this run are
comparable with the B0 run, which also has determinism off.

`worker_init_fn` seeds from `torch.initial_seed()`, which PyTorch redraws from the loader's
generator **every epoch**. With `persistent_workers=False`, that is what makes augmentation
reproducible across runs but genuinely fresh across epochs. Seeding workers from a constant — or
setting an `epoch` attribute on the dataset from the parent process while workers persist — silently
recreates the frozen-augmentation bug, which is the bug the previous TC runs had.

In [ ]:
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
    torch.use_deterministic_algorithms(False)

def worker_init_fn(worker_id):
    # torch.initial_seed() differs per epoch AND per worker -> live augmentation, still seeded
    s = (torch.initial_seed() + worker_id) % (2 ** 32)
    np.random.seed(s); random.seed(s)

set_seed(cfg.SEED)
g = torch.Generator(); g.manual_seed(cfg.SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("seeded:", cfg.SEED, "| device:", DEVICE,
      "| deterministic algorithms: OFF (6x slowdown otherwise)")

## 4. Filename parsing and base-tile grouping

Convention: `image{N}.png` / `image{N}hf.png` / `image{N}vf.png`, masks mirror with the `mask`
prefix. The grouping is needed for two things: the non-legacy split modes, and the leakage audit in
§8 that counts how many of this fold's validation images have a mirror sibling in training.

`records` is built by walking `sorted(os.listdir(IMG_DIR))`, which is **the same ordering the legacy
`SegDataset` used**, so `KFold` indices address the same files as they did in the B0 run. That
identity is what makes the two pairable — do not re-sort `records` anywhere below. Note this differs
from `5__tc-segformer.ipynb`, which re-sorts by `(tile, variant)`; that ordering is fine for a
grouped split but would break `legacy_kfold`.

In [ ]:
IMG_DIR  = Path(cfg.DATA_ROOT) / "images"
MASK_DIR = Path(cfg.DATA_ROOT) / "masks"
assert IMG_DIR.is_dir() and MASK_DIR.is_dir(), f"bad DATA_ROOT: {cfg.DATA_ROOT}"
PAT = re.compile(r"^image(\d+)(hf|vf)?\.png$")

records = []          # (base_id:int, variant:str, image_name:str, mask_name:str)
for f in sorted(os.listdir(IMG_DIR)):          # SAME ORDER the legacy KFold indexed
    m = PAT.match(f)
    assert m, f"unparsed image filename: {f}"
    base_id, variant = int(m.group(1)), (m.group(2) or "orig")
    mask_name = "mask" + f[len("image"):]
    assert (MASK_DIR / mask_name).exists(), f"missing mask for {f}"
    records.append((base_id, variant, f, mask_name))

groups: Dict[int, List[tuple]] = {}
for r in records:
    groups.setdefault(r[0], []).append(r)

BASE_IDS = sorted(groups)
IDX_OF = {r[2]: i for i, r in enumerate(records)}      # image name -> legacy KFold index
assert len(records) == 1224, len(records)
assert len(BASE_IDS) == 408, len(BASE_IDS)
assert all(len(v) == 3 for v in groups.values())

print(f"{len(records)} files | {len(BASE_IDS)} base tiles | id range {BASE_IDS[0]}..{BASE_IDS[-1]}")
print("first 5 in KFold index order:", [r[2] for r in records[:5]])

## 5. Mask colour diagnostic and the class mapping

`(128,0,0)` Residential and `(0,0,128)` River are exact channel reverses, so a BGR/RGB inversion
swaps the dominant class with the rarest one and every downstream metric still looks plausible. The
only reliable detector is a **count** check: Residential must vastly outnumber River.

Anything not in the six-colour table becomes `VOID_LABEL`. `IGNORE_UNLABELLED = True` (the default)
keeps that at 255, excluding those pixels from the loss and from every metric — the same scheme as
the rest of your benchmark and as `5__tc-segformer.ipynb`. Set it to `False` to restore the legacy
behaviour in which black is a trained seventh class.

Either way `metrics_from_cm` reports `mIoU_core6` over the six real classes, so the column you quote
in the paper means the same thing in both modes.

In [ ]:
BGR_TO_CLASS = {
    (128, 0, 0): 0,    # Residential Area
    (0, 128, 0): 1,    # Road
    (0, 0, 128): 2,    # River
    (0, 128, 128): 3,  # Forest
    (128, 128, 0): 4,  # Unused Land
    (128, 0, 128): 5,  # Agricultural Area
}

def pack_bgr(a: np.ndarray) -> np.ndarray:
    a = a.astype(np.uint32)
    return (a[..., 0] << 16) | (a[..., 1] << 8) | a[..., 2]

LUT = np.full(1 << 24, 255, dtype=np.uint8)          # 16 MB; default = unlabelled sentinel
for bgr, idx in BGR_TO_CLASS.items():
    LUT[pack_bgr(np.array(bgr, dtype=np.uint8).reshape(1, 1, 3))[0, 0]] = idx

rng = np.random.RandomState(cfg.SEED)
sample = [records[i] for i in rng.choice(len(records), 40, replace=False)]
colour_hits, class_px = {}, np.zeros(7, dtype=np.int64)

for _, _, _, mname in sample:
    m = cv2.imread(str(MASK_DIR / mname), cv2.IMREAD_COLOR)   # BGR
    cols, cnts = np.unique(m.reshape(-1, 3), axis=0, return_counts=True)
    for c, n in zip(cols, cnts):
        colour_hits[tuple(int(x) for x in c)] = colour_hits.get(tuple(int(x) for x in c), 0) + int(n)
    idx = LUT[pack_bgr(m)]
    class_px += np.bincount(np.where(idx == 255, 6, idx).ravel(), minlength=7)

known = set(BGR_TO_CLASS)
print("distinct BGR colours found in sample:")
for c, n in sorted(colour_hits.items(), key=lambda kv: -kv[1]):
    tag = (f"-> class {BGR_TO_CLASS[c]} ({cfg.ALL_NAMES[BGR_TO_CLASS[c]]})" if c in known
           else f"-> {VOID_LABEL} (Unlabelled)")
    print(f"  BGR{str(c):<16} {n/sum(colour_hits.values())*100:6.2f}%  {tag}")

tot = class_px.sum()
print("\nclass pixel share (sample):")
for i, nm in enumerate(cfg.ALL_NAMES):
    lbl = str(VOID_LABEL) if i == 6 else str(i)
    tail = "  <- excluded from loss and metrics" if (i == 6 and cfg.IGNORE_UNLABELLED) else ""
    print(f"  {lbl:>3} {nm:<14} {class_px[i]/tot*100:6.2f}%{tail}")

assert class_px[0] > class_px[2] * 3, (
    "Residential <= River -- BGR/RGB channel order is probably inverted. STOP and fix.")
print("\nchannel-order check PASSED")

## 6. One-time repack to a 512x512 cache — and the CSA reliability weights

Two caches, deliberately separate.

**Image/label cache** (`CACHE_DIR`) is byte-identical to the one the other baselines use: PNGs at
512, labels written with the `255` sentinel so the cache stays shareable across both class schemes
and with the 6-class notebooks. If you already saved it as a Kaggle dataset, point `PREBUILT_CACHE`
at it and this pass is a no-op.

**CSA cache** (`CSA_CACHE_DIR`) is built only when the recipe needs it, and lives in a separate
directory because `/kaggle/input` is **read-only** — a prebuilt image cache mounted from a Kaggle
dataset cannot receive a third array. Keeping them apart means you can reuse an existing image cache
and still build the weights in `/kaggle/working`, which is the common case.

### The CSA weight, precisely
1. Boundary set `B` = pixels whose 4-neighbour carries a different label — the void region included,
   since the edge of an annotation gap is a real discontinuity in the supervision. The image border
   is **excluded**: frame truncation is not annotation error.
2. `D(x)` = Euclidean distance transform of the complement of `B`: distance to the nearest polygon
   boundary. One EDT for the whole image.
3. Per class, connected components; per component `R = 2 * mean(D)`. For a strip of half-width `r`,
   `D` is roughly uniform on `[0, r]`, so `2*mean(D) = r` — the local half-width. This width
   normalisation is what stops the whole Road class from being driven to `w_min`.
4. `band = clip(tau_c * R, 1 px, CSA_DMAX_M metres)`, then `w(x) = clip(D(x)/band, w_min, 1)`.

The cap is inert for thin classes and binding for area classes — the intended asymmetry. The loop
runs over `range(N_CLASSES)`, so under the default `c6` scheme void pixels are touched by nobody and
keep `w = 1.0`; they are excluded from the loss anyway, and this matches `5__tc-segformer.ipynb`,
which set `W[idx == 255] = 1.0` explicitly. Under `c7` the void becomes class 6 and gets the seventh
`tau`. The two schemes therefore produce **different** weight maps, so the cache marker records which
one built it and refuses to serve the other.

The weight map is the only array resized with `INTER_AREA` after the transform: it is continuous, so
area-averaging is correct, whereas the label map must go through the LUT at **full resolution** and
then `INTER_NEAREST` — resizing the RGB mask first would interpolate between class colours and
manufacture phantom classes at every polygon edge.

Cost: ~25-40 min for the image/label pass, ~10-15 min for the CSA pass, once each. Save both as
Kaggle datasets afterwards and set `PREBUILT_CACHE` / `PREBUILT_CSA_CACHE` to skip them forever.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from scipy import ndimage as ndi

CACHE_INTERP = cv2.INTER_AREA          # legacy used INTER_LINEAR via A.Resize; see section 25
CACHE     = Path(cfg.PREBUILT_CACHE) if cfg.PREBUILT_CACHE else Path(cfg.CACHE_DIR)
CSA_CACHE = Path(cfg.PREBUILT_CSA_CACHE) if cfg.PREBUILT_CSA_CACHE else Path(cfg.CSA_CACHE_DIR)
MARKER, CSA_MARKER = CACHE / "_complete.json", CSA_CACHE / "_csa_complete.json"

# ---------------- pass 1: images + labels ----------------
def _repack_one(rec):
    _, _, iname, mname = rec
    img = cv2.imread(str(IMG_DIR / iname), cv2.IMREAD_COLOR)
    img = cv2.resize(img, (cfg.IMG_SIZE, cfg.IMG_SIZE), interpolation=CACHE_INTERP)
    msk = cv2.imread(str(MASK_DIR / mname), cv2.IMREAD_COLOR)
    lab = LUT[pack_bgr(msk)]                                    # full-res colour -> class index
    lab = cv2.resize(lab, (cfg.IMG_SIZE, cfg.IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    cv2.imwrite(str(CACHE / "images" / iname), img, [cv2.IMWRITE_PNG_COMPRESSION, 3])
    cv2.imwrite(str(CACHE / "labels" / mname), lab, [cv2.IMWRITE_PNG_COMPRESSION, 3])
    return np.bincount(np.where(lab == 255, 6, lab).ravel(), minlength=7)

if MARKER.exists():
    print(f"image/label cache present at {CACHE} ({json.load(open(MARKER))['n']} pairs) -- skipping")
else:
    assert not cfg.PREBUILT_CACHE, f"PREBUILT_CACHE set but {MARKER} missing"
    (CACHE / "images").mkdir(parents=True, exist_ok=True)
    (CACHE / "labels").mkdir(parents=True, exist_ok=True)
    t0 = time.time(); counts = {}
    with ProcessPoolExecutor(max_workers=4) as ex:
        for rec, b in zip(records, ex.map(_repack_one, records, chunksize=8)):
            counts[rec[2]] = b.tolist()
            if len(counts) % 200 == 0:
                el = time.time() - t0
                print(f"  {len(counts)}/{len(records)}  {el:.0f}s  "
                      f"eta {el/len(counts)*(len(records)-len(counts)):.0f}s")
    json.dump({"n": len(counts), "counts": counts}, open(MARKER, "w"))
    print(f"repacked {len(counts)} pairs in {time.time()-t0:.0f}s")

PIX_COUNTS = {k: np.array(v, dtype=np.int64) for k, v in json.load(open(MARKER))["counts"].items()}
print(f"cache: {sum(f.stat().st_size for f in CACHE.rglob('*.png'))/1e9:.2f} GB | "
      f"per-image class counts for {len(PIX_COUNTS)} files (index 6 = unlabelled)")

# ---------------- pass 2: CSA reliability weights ----------------
PROBE_W = cv2.imread(str(MASK_DIR / records[0][3]), cv2.IMREAD_COLOR).shape[1]
PX_PER_M = PROBE_W / cfg.TILE_METRES
DMAX_PX  = cfg.CSA_DMAX_M * PX_PER_M

def csa_weight(idx, taus, wmin, dmax_px, n_classes):
    lab = idx.astype(np.int16)
    b = np.zeros(lab.shape, dtype=bool)
    d = lab[:-1, :] != lab[1:, :]; b[:-1, :] |= d; b[1:, :] |= d
    d = lab[:, :-1] != lab[:, 1:]; b[:, :-1] |= d; b[:, 1:] |= d
    D = cv2.distanceTransform((~b).astype(np.uint8), cv2.DIST_L2, 3)
    W = np.ones(lab.shape, dtype=np.float32)      # anything not in a scored class stays at 1.0
    for c in range(n_classes):
        m = (idx == c).astype(np.uint8)
        if m.sum() == 0:
            continue
        ncc, cc = cv2.connectedComponents(m, connectivity=8)
        if ncc <= 1:
            continue
        means = np.asarray(ndi.mean(D, labels=cc, index=np.arange(1, ncc)), dtype=np.float32)
        R = 2.0 * means                                        # local half-width of the component
        band = np.maximum(np.minimum(taus[c] * R, dmax_px), 1.0)
        bmap = np.zeros(ncc, dtype=np.float32); bmap[1:] = band
        sel = m.astype(bool)
        W[sel] = np.clip(D[sel] / bmap[cc[sel]], wmin, 1.0)
    return W

def _csa_one(rec):
    _, _, iname, mname = rec
    msk = cv2.imread(str(MASK_DIR / mname), cv2.IMREAD_COLOR)
    idx = LUT[pack_bgr(msk)]
    if not cfg.IGNORE_UNLABELLED:
        idx = np.where(idx == 255, 6, idx).astype(np.uint8)     # legacy c7: void is a real class
    W = csa_weight(idx, cfg.CSA_TAU, cfg.CSA_WMIN, DMAX_PX, cfg.N_CLASSES)
    W = np.clip(cv2.resize(W, (cfg.IMG_SIZE, cfg.IMG_SIZE), interpolation=cv2.INTER_AREA), 0, 1)
    cv2.imwrite(str(CSA_CACHE / "relw" / iname), np.round(W * 255).astype(np.uint8),
                [cv2.IMWRITE_PNG_COMPRESSION, 3])
    return float(W.mean())

if not CSA_ENABLE:
    print(f"recipe {cfg.RECIPE}: CSA off -- weight cache not needed, skipping pass 2")
elif CSA_MARKER.exists():
    mk = json.load(open(CSA_MARKER))
    assert (abs(mk["dmax_px"] - DMAX_PX) < 0.5
            and list(mk["tau"])[:cfg.N_CLASSES] == list(cfg.CSA_TAU)[:cfg.N_CLASSES]
            and abs(mk["wmin"] - cfg.CSA_WMIN) < 1e-9
            and mk.get("scheme") == SCHEME_TAG), (
        f"CSA cache was built with different parameters ({mk}); delete {CSA_CACHE} and rebuild")
    print(f"CSA cache present at {CSA_CACHE} ({mk['n']} maps, {mk['scheme']}, "
          f"band {mk['dmax_px']:.1f} px) -- skipping")
else:
    assert not cfg.PREBUILT_CSA_CACHE, f"PREBUILT_CSA_CACHE set but {CSA_MARKER} missing"
    (CSA_CACHE / "relw").mkdir(parents=True, exist_ok=True)
    print(f"native {PROBE_W} px for {cfg.TILE_METRES} m -> {PX_PER_M:.3f} px/m | "
          f"band cap {cfg.CSA_DMAX_M} m = {DMAX_PX:.2f} px "
          f"(identical to the 6.0 m @ 300 m tile used by the original TC runs)")
    t0 = time.time(); done = 0
    with ProcessPoolExecutor(max_workers=4) as ex:
        for _ in ex.map(_csa_one, records, chunksize=8):
            done += 1
            if done % 200 == 0:
                el = time.time() - t0
                print(f"  {done}/{len(records)}  {el:.0f}s  eta {el/done*(len(records)-done):.0f}s")
    json.dump({"n": done, "tau": list(cfg.CSA_TAU), "wmin": cfg.CSA_WMIN, "scheme": SCHEME_TAG,
               "dmax_px": DMAX_PX, "dmax_m": cfg.CSA_DMAX_M, "tile_m": cfg.TILE_METRES,
               "minutes": round((time.time() - t0) / 60, 1)}, open(CSA_MARKER, "w"))
    print(f"CSA pass done in {(time.time()-t0)/60:.1f} min -> {CSA_CACHE}")

## 7. Load the cache into RAM

1224 x (512x512x3 + 512x512 [+ 512x512]) uint8 = ~1.3 GB without CSA, ~1.6 GB with it, out of the
~13 GB available. Preloading removes disk I/O from the training loop entirely, which matters because
the augmentation is cheap and two workers would otherwise be the bottleneck.

Labels keep the `255` sentinel under the default scheme and are remapped to class 6 only when
`IGNORE_UNLABELLED = False`, so the on-disk cache stays valid for both. The assert is the guard that
matters most here: a stray 255 reaching a 6-way loss that was not given `ignore_index` is a
device-side assert thousands of steps into training, not a clean error.

In [ ]:
RAM = {}
t0 = time.time()
for i, (_, _, iname, mname) in enumerate(records):
    if cfg.PRELOAD_RAM:
        img = cv2.imread(str(CACHE / "images" / iname), cv2.IMREAD_COLOR)
        lab = cv2.imread(str(CACHE / "labels" / mname), cv2.IMREAD_GRAYSCALE)
        if not cfg.IGNORE_UNLABELLED:
            lab = np.where(lab == 255, 6, lab).astype(np.uint8)      # legacy 7-class remap
        if CSA_ENABLE:
            w = cv2.imread(str(CSA_CACHE / "relw" / iname), cv2.IMREAD_GRAYSCALE)
            assert w is not None, f"missing CSA map for {iname}"
        else:
            w = None
        RAM[iname] = (img, lab, w)
        if (i + 1) % 400 == 0: print(f"  loaded {i+1}/{len(records)}")

if cfg.PRELOAD_RAM:
    mb = sum(a.nbytes + b.nbytes + (0 if c is None else c.nbytes) for a, b, c in RAM.values()) / 1e6
    print(f"preloaded {len(RAM)} pairs, {mb:.0f} MB, {time.time()-t0:.0f}s")
    u = np.unique(np.concatenate([b[::32, ::32].ravel() for _, b, _ in RAM.values()]))
    allowed = set(range(cfg.N_CLASSES)) | ({cfg.IGNORE_INDEX} if cfg.IGNORE_UNLABELLED else set())
    assert set(u.tolist()) <= allowed, f"unexpected label values: {u}"
    print("label value check PASSED:", u.tolist())
    if CSA_ENABLE:
        ws = np.concatenate([c[::32, ::32].ravel() for _, _, c in RAM.values()]).astype(np.float32) / 255.0
        print(f"CSA weight map: mean {ws.mean():.3f} | min {ws.min():.3f} | "
              f"frac at full weight {np.mean(ws > 0.999):.3f}")
else:
    print("PRELOAD_RAM off -- reading from disk per item")

## 8. Splitters — legacy KFold over images, plus the grouped controls

### `legacy_kfold` (default)
`KFold(n_splits=5, shuffle=True, random_state=42).split(dataset)` over the 1224 **image indices**,
taking the `FOLD`-th split. `records` is in the same order the legacy `SegDataset` used, so the
indices address the same files as they did in the B0 run — which is what makes the two pairable.

### The leakage audit
The cell prints how many validation images have at least one of their own flip variants in the
training set. Under `legacy_kfold` expect ~96%: a held-out image whose left-right mirror was a
training example is not a held-out image. Under `random` and `block` it must print exactly 0, and
the assert enforces that so a future edit cannot reintroduce the problem silently.

Read the printed number as the single most important line in this notebook. Whatever mIoU comes out
the other end has to be interpreted against it, and it is the reason §21 reports the TC-minus-B0
delta rather than TC's absolute score.

In [ ]:
from sklearn.model_selection import KFold

def legacy_kfold_split(n_items: int, fold: int, n_folds: int, seed: int):
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for i, (tr, va) in enumerate(kf.split(np.arange(n_items))):
        if i == fold:
            return list(tr), list(va)
    raise IndexError(fold)

def block_split(base_ids: List[int], fold: int, n_folds: int, buffer: int):
    n = len(base_ids)
    edges = [round(i * n / n_folds) for i in range(n_folds + 1)]
    lo, hi = edges[fold], edges[fold + 1]
    val = base_ids[lo:hi]
    train = base_ids[:max(0, lo - buffer)] + base_ids[min(n, hi + buffer):]
    return train, val

def random_split(base_ids: List[int], fold: int, n_folds: int, seed: int):
    ids = list(base_ids)
    random.Random(seed).shuffle(ids)
    edges = [round(i * len(ids) / n_folds) for i in range(n_folds + 1)]
    val = ids[edges[fold]:edges[fold + 1]]
    train = ids[:edges[fold]] + ids[edges[fold + 1]:]
    return sorted(train), sorted(val)

def expand(ids): return [r for i in ids for r in groups[i]]

if cfg.SPLIT_MODE == "legacy_kfold":
    tr_i, va_i = legacy_kfold_split(len(records), cfg.FOLD, cfg.N_FOLDS, cfg.KFOLD_SEED)
    tr_recs, va_recs = [records[i] for i in tr_i], [records[i] for i in va_i]
elif cfg.SPLIT_MODE == "block":
    tr_ids, va_ids = block_split(BASE_IDS, cfg.FOLD, cfg.N_FOLDS, cfg.BLOCK_BUFFER)
    tr_recs, va_recs = expand(tr_ids), expand(va_ids)
else:
    tr_ids, va_ids = random_split(BASE_IDS, cfg.FOLD, cfg.N_FOLDS, cfg.SEED)
    tr_recs, va_recs = expand(tr_ids), expand(va_ids)

tr_bases, va_bases = {r[0] for r in tr_recs}, {r[0] for r in va_recs}
tr_names = {r[2] for r in tr_recs}
shared_tiles = tr_bases & va_bases
leaky = [r for r in va_recs if any(s[2] in tr_names for s in groups[r[0]] if s[2] != r[2])]

print(f"{cfg.SPLIT_MODE} fold {cfg.FOLD}/{cfg.N_FOLDS}: "
      f"{len(tr_recs)} train images / {len(va_recs)} val images")
print(f"  base tiles: {len(tr_bases)} train / {len(va_bases)} val / {len(shared_tiles)} in BOTH")
print(f"  VARIANT LEAKAGE: {len(leaky)}/{len(va_recs)} val images "
      f"({len(leaky)/max(len(va_recs),1)*100:.1f}%) have a mirror of themselves in train")

if cfg.SPLIT_MODE != "legacy_kfold":
    assert not shared_tiles and not leaky, "grouped split must not leak variants"
    print("  grouped split: no variant leakage (as required)")

## 9. Class weights — legacy inverse frequency over the whole dataset

The legacy script does this:

```
class_weights = 1.0 / (class_counts + 1e-6)
class_weights = class_weights / class_weights.sum() * num_classes
```

with `class_counts` accumulated by iterating **`dataset`**, not `Subset(dataset, train_idx)`. Two
consequences, both reproduced here under `WEIGHT_SOURCE = "all"`: validation class priors enter the
loss, and the weights are raw inverse frequency with no cap. Over the 6 real classes the spread is
93:1, so River's gradient is amplified about ninety-fold relative to Residential. Counts run over
the scored classes only, so under the default scheme the unlabelled blobs no longer draw the largest
weight of all — which is one of the reasons the B0 notebook moved to `c6`.

This replaces the effective-number scheme with cap 10 that `5__tc-segformer.ipynb` uses, but the two
are closer than they look. At pixel-scale counts `1 - beta^n` collapses to `n/N` for every class
except the dominant one, so effective number *becomes* inverse frequency: the two agree within 5% on
Road, River, Forest, UnusedLand and Agricultural and differ only on Residential. The `W_CAP = 10` in
the TC notebook never binds — the spread tops out at 71.6 on its own.

**The one place this does bite.** The TC notebook's markdown credits that cap with stabilising the
6e5 gradient norms seen in earlier runs, and this notebook has both uncapped weights *and* no
gradient clipping, which is the legacy configuration. Plain CE survived it in the B0 run; adding
Lovasz on top under `RECIPE = "tc_full"` has not been tested in that combination. `grad_norm_true`
is printed every epoch: if it sits above ~50 for several epochs while `train_ce` stalls or goes NaN,
set `CLIP_NORM = 1.0` and restart the fold. That is a deviation worth taking and worth recording,
not a silent default.

The balanced sampler draws on tile **presence** frequency over the six real classes only.

In [ ]:
def counts_over(recs):
    n = np.zeros(cfg.N_CLASSES, dtype=np.float64)
    for _, _, iname, _ in recs:
        n += PIX_COUNTS[iname][:cfg.N_CLASSES]
    return n

def legacy_inv_freq_weights(n):
    w = 1.0 / (n + 1e-6)
    return w / w.sum() * cfg.N_CLASSES          # literal transcription

def effective_number_weights(n, cap):
    beta = 1.0 - 1.0 / n.sum()
    eff = (1.0 - np.power(beta, n)) / (1.0 - beta)
    w = 1.0 / np.maximum(eff, 1e-12)
    w = w / w.mean()
    if cap: w = np.minimum(w, cap)
    return w / w.mean()

SRC = records if cfg.WEIGHT_SOURCE == "all" else tr_recs
COUNTS = counts_over(SRC)

if cfg.CLASS_WEIGHT_MODE == "legacy_inv_freq":
    CLASS_W = legacy_inv_freq_weights(COUNTS)
elif cfg.CLASS_WEIGHT_MODE == "effective_number":
    CLASS_W = effective_number_weights(COUNTS, cfg.EN_BETA_CAP)
elif cfg.CLASS_WEIGHT_MODE == "none":
    CLASS_W = np.ones(cfg.N_CLASSES, dtype=np.float64)
else:
    raise ValueError(cfg.CLASS_WEIGHT_MODE)
if cfg.WEIGHT_CAP:
    CLASS_W = np.minimum(CLASS_W, cfg.WEIGHT_CAP)

print(f"mode: {cfg.CLASS_WEIGHT_MODE} | counted over: {cfg.WEIGHT_SOURCE} "
      f"({len(SRC)} images, {COUNTS.sum():.3e} px across {cfg.N_CLASSES} scored classes)")
print(f"{'class':<14}{'px %':>10}{'weight':>12}")
for i, nm in enumerate(cfg.CLASS_NAMES):
    print(f"{nm:<14}{COUNTS[i]/COUNTS.sum()*100:9.3f}%{CLASS_W[i]:12.4f}")
print(f"weight ratio max/min: {CLASS_W.max()/CLASS_W.min():.1f}")
CLASS_W_T = torch.tensor(CLASS_W, dtype=torch.float32).to(DEVICE)

# ---- tile-presence frequency over the six real classes, for the balanced sampler ----
PRESENCE = np.stack([PIX_COUNTS[r[2]][:6] > 0 for r in records])       # (1224, 6)
TR_ROWS = np.array([IDX_OF[r[2]] for r in tr_recs])
tile_presence_freq = PRESENCE[TR_ROWS].mean(0)
print("\ntile-presence frequency (train):",
      {n: f"{v*100:.1f}%" for n, v in zip(cfg.ALL_NAMES[:6], tile_presence_freq)})

## 10. Dataset and DataLoaders — the legacy albumentations pipeline, carrying the CSA map

The legacy transform, verbatim:

```
A.Compose([A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
           A.RandomRotate90(p=0.5), A.Resize(512, 512), ToTensorV2()])
```

Three things about it are worth stating plainly rather than quietly fixing:

- **The flips are applied on top of flips that are already in the data.** `imageNhf.png` and
  `imageNvf.png` are materialised mirrors, so a random hflip at load time can turn one file into an
  exact copy of another file sitting in the same epoch. `5__tc-segformer.ipynb` drops them for
  exactly this reason and augments with rot90 only. Reproduced as-is here, following the B0 run.
- **The same transform ran on validation in the legacy script.** `Subset(dataset, val_idx)` shares
  one `SegDataset`, so val images were randomly flipped and rotated at scoring time. In expectation
  that costs little on D4-symmetric nadir imagery, but it makes the val score non-deterministic
  between epochs, which makes checkpoint selection partly a lottery over augmentation draws.
  `VAL_USES_TRAIN_TF` is therefore **`False`** by default; set it `True` for exact parity. §19 scores
  through the clean loader regardless, so the reported number was never affected — only the choice of
  which epoch to keep.
- **`A.Resize` after the flips is a no-op** on a 512 cache, and albumentations resizes masks with
  nearest regardless, so labels are safe either way.

**The CSA map rides along as a second mask target.** `additional_targets={"relw": "mask"}` puts it
through the identical geometric ops as the label. That is exact here and not an approximation: the
only geometric transforms in the pipeline are flips and rot90, which are pure index permutations, so
no resampling happens and nearest-neighbour handling of a continuous field costs nothing. If you ever
add scale jitter or rotation by arbitrary angles this stops being true and the weight map needs its
own bilinear target.

`num_workers=2`, `persistent_workers=False`, `drop_last=False` are the legacy values. The second one
is not cosmetic: with `worker_init_fn` seeding from `torch.initial_seed()`, it is what keeps
augmentation live across epochs, and it is the fix for the frozen-augmentation bug the previous TC
runs had.

**The sampler replaces `shuffle=True` when the recipe asks for it.** Each tile is weighted by the
inverse tile-presence frequency of the rarest real class it contains, capped at `SAMPLER_CAP`, and
`len(train)` samples are drawn with replacement — epoch length and effective batch are unchanged, but
rare-class gradient becomes dense instead of spiky. Caveat for the paper: with replacement, an
"epoch" no longer visits every tile exactly once.

In [ ]:
from transformers import SegformerImageProcessor

processor = SegformerImageProcessor.from_pretrained(cfg.HF_MODEL_ID)
PROC_MEAN = np.array(processor.image_mean, np.float32).reshape(3, 1, 1)
PROC_STD  = np.array(processor.image_std,  np.float32).reshape(3, 1, 1)

train_tf = A.Compose([
    A.HorizontalFlip(p=cfg.AUG_HFLIP_P),
    A.VerticalFlip(p=cfg.AUG_VFLIP_P),
    A.RandomRotate90(p=cfg.AUG_ROT90_P),
    A.Resize(cfg.IMG_SIZE, cfg.IMG_SIZE),
    ToTensorV2(),
], additional_targets={"relw": "mask"})
clean_tf = A.Compose([A.Resize(cfg.IMG_SIZE, cfg.IMG_SIZE), ToTensorV2()],
                     additional_targets={"relw": "mask"})

class SatSegLegacy(Dataset):
    # Returns (image, label, csa_weight). The weight is all-ones when CSA is off, so the loss,
    # the core-region metric and the checkpoint all have exactly one code path.
    def __init__(self, recs, tf):
        self.recs, self.tf = recs, tf

    def __len__(self): return len(self.recs)

    def _read(self, iname, mname):
        if cfg.PRELOAD_RAM:
            img, lab, w = RAM[iname]
            return img.copy(), lab.copy(), (None if w is None else w.copy())
        img = cv2.imread(str(CACHE / "images" / iname), cv2.IMREAD_COLOR)
        lab = cv2.imread(str(CACHE / "labels" / mname), cv2.IMREAD_GRAYSCALE)
        if not cfg.IGNORE_UNLABELLED:
            lab = np.where(lab == 255, 6, lab).astype(np.uint8)
        w = cv2.imread(str(CSA_CACHE / "relw" / iname), cv2.IMREAD_GRAYSCALE) if CSA_ENABLE else None
        return img, lab, w

    def __getitem__(self, i):
        _, _, iname, mname = self.recs[i]
        img, lab, w = self._read(iname, mname)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)            # legacy: PIL .convert("RGB")
        if w is None:
            w = np.full(lab.shape, 255, np.uint8)
        out = self.tf(image=img, mask=lab, relw=w)
        x, y = out["image"], out["mask"].long()               # x is uint8 CHW
        rw = out["relw"].float() / 255.0
        if cfg.PREPROCESS == "processor":
            x = processor(images=x, return_tensors="pt")["pixel_values"].squeeze(0)
        else:
            x = torch.from_numpy((x.float().numpy() / 255.0 - PROC_MEAN) / PROC_STD)
        return x, y, rw

def make_sampler(recs, presence, tile_freq, cap):
    inv = 1.0 / np.maximum(tile_freq, 1e-6)
    rows = [IDX_OF[r[2]] for r in recs]
    w = np.array([max([inv[c] for c in range(6) if presence[j, c]], default=1.0) for j in rows],
                 dtype=np.float64)
    w = np.minimum(w / w.min(), cap)
    return WeightedRandomSampler(torch.as_tensor(w), num_samples=len(rows), replacement=True)

train_ds = SatSegLegacy(tr_recs, train_tf)
val_ds   = SatSegLegacy(va_recs, train_tf if cfg.VAL_USES_TRAIN_TF else clean_tf)
clean_ds = SatSegLegacy(va_recs, clean_tf)
sampler  = make_sampler(tr_recs, PRESENCE, tile_presence_freq, cfg.SAMPLER_CAP) if BALANCED_SAMPLER else None

train_ld = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, sampler=sampler,
                      shuffle=(sampler is None), drop_last=cfg.DROP_LAST,
                      num_workers=cfg.NUM_WORKERS, pin_memory=True,
                      persistent_workers=cfg.PERSISTENT_WORKERS,
                      worker_init_fn=worker_init_fn, generator=g)
val_ld   = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                      num_workers=cfg.NUM_WORKERS, pin_memory=True,
                      persistent_workers=cfg.PERSISTENT_WORKERS, worker_init_fn=worker_init_fn)
clean_ld = DataLoader(clean_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                      num_workers=cfg.NUM_WORKERS, pin_memory=True)

# processor vs manual: verify they are the same arithmetic, not merely similar
_img0 = (RAM[tr_recs[0][2]][0] if cfg.PRELOAD_RAM
         else cv2.imread(str(CACHE / "images" / tr_recs[0][2]), cv2.IMREAD_COLOR))
_raw = torch.from_numpy(np.ascontiguousarray(
    cv2.cvtColor(_img0, cv2.COLOR_BGR2RGB).transpose(2, 0, 1)))
_a = processor(images=_raw, return_tensors="pt")["pixel_values"].squeeze(0).numpy()
_b = (_raw.float().numpy() / 255.0 - PROC_MEAN) / PROC_STD
assert np.abs(_a - _b).max() < 1e-4, np.abs(_a - _b).max()
print(f"processor == manual normalisation (max abs diff {np.abs(_a-_b).max():.2e})")

xb, yb, wb = next(iter(train_ld))
_allowed = set(range(cfg.N_CLASSES)) | ({cfg.IGNORE_INDEX} if cfg.IGNORE_UNLABELLED else set())
assert set(yb.unique().tolist()) <= _allowed, yb.unique().tolist()
print(xb.shape, xb.dtype, yb.shape, yb.dtype, wb.shape, "| label values:", torch.unique(yb).tolist())
print(f"CSA weight in batch: min {float(wb.min()):.3f} max {float(wb.max()):.3f} "
      f"mean {float(wb.mean()):.3f} (all 1.0 when CSA is off)")
print(f"steps/epoch: {len(train_ld)} | val batches: {len(val_ld)} | "
      f"sampler={'balanced' if sampler else 'shuffle'} | "
      f"val tf={'train (legacy)' if cfg.VAL_USES_TRAIN_TF else 'clean'} | "
      f"persistent_workers={cfg.PERSISTENT_WORKERS} (live augmentation)")

## 11. Loss — weighted CE (x CSA) + Lovasz (CSA-masked, ramped) + soft-clDice

The `RECIPE` from §2 decides which terms are live. Under `"legacy_pure"` the whole thing collapses to
`nn.CrossEntropyLoss(weight=class_weights, ignore_index=255)` with the CSA map at all-ones — exactly
the B0 objective. Under `"tc_full"` all three terms are on.

**The CE term is normalised by the sum of applied weights, not by the pixel count.** This is the one
place where `5__tc-segformer.ipynb` and the legacy script could not both be followed, and the
difference is not cosmetic. `nn.CrossEntropyLoss(weight=w)` returns `sum(w_y * ce) / sum(w_y)` over
the non-ignored pixels; the original TC loss returned `sum(w_y * relw * ce) / sum(relw)`, i.e. it
divided by the pixel count. Those agree when the class weights average 1 over pixels, which is true
of the capped effective-number weights the TC notebook used — and false of the legacy `1/n` weights,
which Residential dominates. At this dataset's class shares the pixel-mean weight is roughly 1/8, so
the TC normalisation would have produced a CE about **8x smaller**, and with a flat 3e-5 and no
scheduler that is an 8x smaller effective learning rate on the CE term — enough on its own to make a
recipe comparison look like a failed recipe. The denominator here is `sum(cw[target] * relw)` over
valid pixels, which folds CSA in and reduces exactly to the legacy call when the weight map is all
ones. The cell asserts that identity on every run rather than asking you to trust it.

Four design decisions carried over from `5__tc-segformer.ipynb` unchanged:

**1. The whole loss is fp32, outside autocast.** Lovasz sorts per-pixel errors and its gradient is a
cumsum difference, which accumulates visible error and can NaN in fp16. The legacy script computed
its CE inside autocast, which changes nothing because `cross_entropy` is on PyTorch's fp32 autocast
list and is promoted automatically — so this is the same arithmetic stated explicitly.

**2. CSA enters CE as a per-pixel weight but Lovasz as a hard mask.** Lovasz is a set-function
surrogate over the *sorted* error vector; there is no principled way to scale individual pixels
inside it, and it concentrates gradient exactly where errors cluster, i.e. at boundaries — so a
weighted CE and an unweighted Lovasz would actively fight each other. Pixels with
`w < CSA_LOVASZ_THRESH` are folded into the ignore set for the Lovasz term only.

**3. Lovasz is ramped from 0 over the first 10 epochs.** Its gradient is defined by the error
*ranking*; before the network can rank anything it emits large near-random updates — which is exactly
when the freshly initialised detail-path gate is deciding whether to open.

**4. soft-clDice** (Shit et al., CVPR 2021) applies only to Road and River via iterative soft
erosion/opening, costs no parameters, runs at 256 px, and is skipped per-sample when the class is
absent from ground truth — scoring absent classes would inflate the River term.

`ignore_index` pixels are excluded from all three terms. **Read CE, not the total**: the printed total
is `CE + lambda*Lovasz + mu*clDice` and both multipliers ramp, so the total is non-stationary by
construction and a rising loss during the ramp means nothing. Only `train_ce` / `val_ce` are
comparable across epochs.

In [ ]:
def lovasz_grad(gt_sorted):
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    inter = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jacc = 1.0 - inter / union
    if p > 1:
        jacc[1:p] = jacc[1:p] - jacc[0:-1]
    return jacc

def lovasz_softmax_flat(probs, labels, classes="present"):
    if probs.numel() == 0:
        return probs.sum() * 0.0
    losses = []
    for c in range(probs.size(1)):
        fg = (labels == c).float()
        if classes == "present" and fg.sum() == 0:
            continue
        errors = (fg - probs[:, c]).abs()
        errors_sorted, perm = torch.sort(errors, 0, descending=True)
        losses.append(torch.dot(errors_sorted, lovasz_grad(fg[perm])))
    if not losses:
        return probs.sum() * 0.0
    return torch.stack(losses).mean()

def soft_erode(x):
    p1 = -F.max_pool2d(-x, (3, 1), (1, 1), (1, 0))
    p2 = -F.max_pool2d(-x, (1, 3), (1, 1), (0, 1))
    return torch.min(p1, p2)

def soft_open(x):
    return F.max_pool2d(soft_erode(x), (3, 3), (1, 1), (1, 1))

def soft_skel(x, iters):
    x1 = soft_open(x); skel = F.relu(x - x1)
    for _ in range(iters):
        x = soft_erode(x); x1 = soft_open(x)
        delta = F.relu(x - x1)
        skel = skel + F.relu(delta - skel * delta)
    return skel

def soft_cldice(prob_c, gt_c, iters, eps=1.0):
    sp, sg = soft_skel(prob_c, iters), soft_skel(gt_c, iters)
    tprec = (torch.sum(sp * gt_c, (1, 2, 3)) + eps) / (torch.sum(sp, (1, 2, 3)) + eps)
    tsens = (torch.sum(sg * prob_c, (1, 2, 3)) + eps) / (torch.sum(sg, (1, 2, 3)) + eps)
    return 1.0 - 2.0 * tprec * tsens / (tprec + tsens)

class GeoLoss(nn.Module):
    def __init__(self, cfg, class_w):
        super().__init__()
        self.cfg = cfg
        self.register_buffer("cw", class_w)
        self.lam = 0.0; self.mu = 0.0

    def set_epoch(self, e):
        c = self.cfg
        self.lam = (c.LAMBDA_LOVASZ * min(1.0, (e + 1) / max(c.LOVASZ_WARMUP_EPOCHS, 1))
                    if LOVASZ_ENABLE else 0.0)
        self.mu = c.CLDICE_MU if (CLDICE_ENABLE and e >= c.CLDICE_START_EPOCH) else 0.0

    def forward(self, logits, target, relw):
        c = self.cfg
        logits = logits.float()                                      # fp32, outside autocast
        valid = (target != c.IGNORE_INDEX)
        tgt = torch.where(valid, target, torch.zeros_like(target))   # 255 would index out of range
        vf = valid.float()

        ce_map = F.cross_entropy(logits, tgt, weight=self.cw,
                                 reduction="none", label_smoothing=c.LABEL_SMOOTHING)
        # Normalise by the SUM OF APPLIED WEIGHTS, not by the pixel count: ce_map already carries
        # cw[target], so this is the weighted mean and reduces EXACTLY to
        # nn.CrossEntropyLoss(weight=w, ignore_index=255) when relw is all ones. See the markdown.
        wmap = self.cw[tgt] * relw * vf
        ce = (ce_map * relw * vf).sum() / wmap.sum().clamp(min=1e-6)

        probs = logits.softmax(1)

        lv = logits.sum() * 0.0
        if self.lam > 0:
            keep = valid & (relw >= c.CSA_LOVASZ_THRESH)             # hard mask, not a weight
            if keep.sum() > 0:
                lv = lovasz_softmax_flat(probs.permute(0, 2, 3, 1)[keep], target[keep])

        cl = logits.sum() * 0.0
        if self.mu > 0:
            r = c.CLDICE_RES
            pr = F.adaptive_avg_pool2d(probs, r)
            terms = []
            for ci in c.CLDICE_CLASSES:
                gt = F.adaptive_avg_pool2d(((target == ci) & valid).float().unsqueeze(1), r)
                present = gt.sum((1, 2, 3)) > 0
                if present.any():
                    terms.append(soft_cldice(pr[:, ci:ci + 1][present], gt[present],
                                             c.CLDICE_ITERS).mean())
            if terms:
                cl = torch.stack(terms).mean()

        total = ce + self.lam * lv + self.mu * cl
        return total, dict(ce=float(ce.detach()), lovasz=float(lv.detach()),
                           cldice=float(cl.detach()), lam=self.lam, mu=self.mu)

criterion = GeoLoss(cfg, CLASS_W_T).to(DEVICE)

# Guard: with an all-ones weight map the CE term must reduce EXACTLY to the legacy call.
# This is what keeps "legacy recipe" an assertion rather than a claim.
_lo = torch.randn(2, cfg.N_CLASSES, 64, 64, device=DEVICE)
_tg = torch.randint(0, cfg.N_CLASSES, (2, 64, 64), device=DEVICE)
if cfg.IGNORE_UNLABELLED:
    _tg[:, :8, :8] = cfg.IGNORE_INDEX                 # exercise the ignore path too
_lam, _mu = criterion.lam, criterion.mu
criterion.lam = criterion.mu = 0.0
_mine, _ = criterion(_lo, _tg, torch.ones_like(_tg, dtype=torch.float32))
_ref = nn.CrossEntropyLoss(weight=CLASS_W_T, ignore_index=cfg.IGNORE_INDEX)(_lo, _tg)
criterion.lam, criterion.mu = _lam, _mu
assert abs(float(_mine) - float(_ref)) < 1e-5, (float(_mine), float(_ref))
print(f"CE parity check PASSED: {float(_mine):.6f} == "
      f"nn.CrossEntropyLoss(weight=w, ignore_index={cfg.IGNORE_INDEX}) {float(_ref):.6f}")

print(f"loss: weighted CE x CSA{' + Lovasz' if LOVASZ_ENABLE else ''}"
      f"{' + soft-clDice' if CLDICE_ENABLE else ''} | ignore_index={cfg.IGNORE_INDEX}"
      f"{'' if cfg.IGNORE_UNLABELLED else ' (inert: no pixel is 255 in the c7 scheme)'}")
print(f"  lovasz lambda target {cfg.LAMBDA_LOVASZ if LOVASZ_ENABLE else 0.0} "
      f"(ramp {cfg.LOVASZ_WARMUP_EPOCHS} epochs) | clDice mu "
      f"{cfg.CLDICE_MU if CLDICE_ENABLE else 0.0} (from epoch {cfg.CLDICE_START_EPOCH})")
print("  weights:", dict(zip(cfg.CLASS_NAMES, np.round(CLASS_W, 4).tolist())))

## 12. Model — TC-SegFormer — and the optimizer

MiT-B0 encoder + the pretrained all-MLP decode head, with a residual H/2 tail bolted on:

```
image 512 --> MiT encoder --> 4 stages --> linear_c + linear_fuse --> fused (D, 128, 128)
   |                                                                     |
   |                                                        1x1 proj to 64 ch
   |                                                                     |
   +--> detail path (3->8->16->32, stride 2, GroupNorm) --> (32, 256, 256)
                                        |                                |
                                        +---- gate * detail --concat-----+
                                                       |
                                              3x3 fuse -> 64 -> 1x1 refine -> residual @ H/2 -> x2
```

Unchanged from `5__tc-segformer.ipynb`. Three properties worth restating because they matter for how
this run should be read:

*The refinement is a residual, not a replacement.* The base path goes straight through
`head.classifier` exactly as in plain SegFormer, and the H/2 branch adds a correction whose final
conv is initialised at `std = 1e-4`. At step 0 the output is vanilla SegFormer-B0 to four decimal
places, so `USE_DETAIL = False` reproduces the B0 run exactly rather than approximately — and this
run should track the B0 curve from the start and then diverge.

*GroupNorm, not BatchNorm.* SegFormer is LayerNorm throughout; adding BN at batch 16 invites a
train/eval statistics discrepancy the rest of the network does not have.

*Overhead.* ~0.08 M params on 3.72 M (+2%) and ~1.3 GFLOPs. A few parameters operating at high
spatial resolution is a comparatively overfitting-resistant place to add capacity on a 408-tile
dataset, unlike widening the encoder — which is why `DETAIL_CH` stays at (8, 16, 32).

### The optimizer, and the one place parity had to give

The legacy script is `AdamW(model.parameters(), lr=3e-5)`: one group, flat, `weight_decay = 0.01`
from torch's default applied to every tensor including LayerNorm gains and biases, no scheduler, no
warmup, no clipping, no EMA. All of that is reproduced.

What cannot be reproduced without breaking the model is a single flat LR across parameters that do
not exist in B0. `refine` starts at `std = 1e-4` *by design*; at 3e-5 with no warmup it would take a
large fraction of the run just to leave its initialisation, and TC would converge to B0 plus noise.
`LR_MODE = "new_boosted"` therefore holds the encoder and the pretrained decode head at exactly 3e-5
and gives `proj` / `detail` / `fuse` / `refine` / `gate` 10x that — the same new:pretrained ratio the
original TC notebook used (6e-4 vs 6e-5). `"legacy_flat"` puts everything at 3e-5.

**The gate value is the diagnostic.** It is printed every epoch. If it stays within a few percent of
`DETAIL_GATE_INIT = 0.1` for 30+ epochs, the detail path is not being used and the run is B0 with
extra FLOPs — that is the signal to check `LR_MODE`, not to keep training.

### The resolution trap

Plain SegFormer emits logits at H/4 and the legacy script upsamples with
`F.interpolate(..., mode="bilinear", align_corners=False)` before the loss. TC returns **full-
resolution logits already**, so `forward_logits` upsamples only if it has to. The probe assert below
is the check never to skip: if it prints 128, every metric in this notebook is computed at the wrong
resolution.

In [ ]:
from transformers import SegformerConfig, SegformerForSemanticSegmentation

class DetailPath(nn.Module):
    def __init__(self, chs):
        super().__init__()
        c1, c2, c3 = chs
        def blk(i, o, s): return nn.Sequential(nn.Conv2d(i, o, 3, s, 1, bias=False),
                                               nn.GroupNorm(min(8, o), o), nn.ReLU(inplace=True))
        self.net = nn.Sequential(blk(3, c1, 2), blk(c1, c2, 1), blk(c2, c3, 1))
        self.out_ch = c3
    def forward(self, x): return self.net(x)

class TCSegFormer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        conf = SegformerConfig.from_pretrained(cfg.HF_MODEL_ID)
        conf.num_labels = cfg.N_CLASSES
        if cfg.DROP_PATH is not None:            # legacy passed none -> checkpoint defaults
            conf.drop_path_rate = cfg.DROP_PATH
        base = SegformerForSemanticSegmentation.from_pretrained(
            cfg.HF_MODEL_ID, config=conf, ignore_mismatched_sizes=True)
        self.encoder = base.segformer
        self.head = base.decode_head
        D = conf.decoder_hidden_size
        self.use_detail = cfg.USE_DETAIL
        if self.use_detail:
            self.proj = nn.Conv2d(D, cfg.FUSE_CH, 1)
            self.detail = DetailPath(cfg.DETAIL_CH)
            self.gate = nn.Parameter(torch.tensor(float(cfg.DETAIL_GATE_INIT)))
            self.fuse = nn.Sequential(
                nn.Conv2d(cfg.FUSE_CH + self.detail.out_ch, cfg.FUSE_CH, 3, 1, 1, bias=False),
                nn.GroupNorm(8, cfg.FUSE_CH), nn.ReLU(inplace=True))
            self.refine = nn.Conv2d(cfg.FUSE_CH, cfg.N_CLASSES, 1)
            # near-zero init: at step 0 the refinement contributes ~nothing, so the model IS
            # vanilla SegFormer. Small-normal rather than exactly zero so that proj/fuse/detail
            # receive gradient from the very first step instead of waiting one update.
            nn.init.normal_(self.refine.weight, std=cfg.REFINE_INIT_STD)
            nn.init.zeros_(self.refine.bias)

    def fused_feature(self, hidden_states):
        # SegformerMLP flattens (B,C,H,W) -> (B,HW,D) internally, hence the permute/reshape back
        bs = hidden_states[-1].shape[0]
        outs = []
        for hs, mlp in zip(hidden_states, self.head.linear_c):
            h, w = hs.shape[2], hs.shape[3]
            x = mlp(hs).permute(0, 2, 1).reshape(bs, -1, h, w)
            x = F.interpolate(x, size=hidden_states[0].shape[2:], mode="bilinear", align_corners=False)
            outs.append(x)
        x = self.head.linear_fuse(torch.cat(outs[::-1], dim=1))
        x = self.head.batch_norm(x)
        x = self.head.activation(x)
        return self.head.dropout(x)

    def forward(self, x):
        H, W = x.shape[-2:]
        hs = self.encoder(x, output_hidden_states=True, return_dict=True).hidden_states
        fused = self.fused_feature(hs)                              # (B, D, H/4, W/4)
        base = self.head.classifier(fused)                          # vanilla SegFormer logits @ H/4
        logits = F.interpolate(base, size=(H, W), mode="bilinear", align_corners=False)
        if not self.use_detail:
            return logits                                           # == plain SegFormer, exactly
        f = F.interpolate(self.proj(fused), size=(H // 2, W // 2), mode="bilinear", align_corners=False)
        f = torch.cat([f, self.gate * self.detail(x)], dim=1)
        r = self.refine(self.fuse(f))                               # (B, C, H/2, W/2) residual
        return logits + F.interpolate(r, size=(H, W), mode="bilinear", align_corners=False)

NEW_PREFIX = ("proj", "detail", "fuse", "refine", "gate")

def build_model():
    return TCSegFormer(cfg)

def forward_logits(m, x):
    lo = m(x)
    if lo.shape[-2:] != x.shape[-2:]:                               # true only if USE_DETAIL=False
        lo = F.interpolate(lo, size=x.shape[-2:], mode="bilinear", align_corners=False)
    return lo

model = build_model().to(DEVICE)

# ---- optimizer: legacy single group, or legacy + a 10x tier for the new modules only ----
def build_optimizer(m, cfg):
    if cfg.LR_MODE == "legacy_flat":
        return torch.optim.AdamW(m.parameters(), lr=cfg.LR)         # literal legacy call
    new, old = [], []
    for n, p in m.named_parameters():
        if not p.requires_grad: continue
        (new if n.split(".")[0] in NEW_PREFIX else old).append(p)
    return torch.optim.AdamW([dict(params=old, lr=cfg.LR),
                              dict(params=new, lr=cfg.LR * cfg.LR_NEW_MULT)])

optimizer = build_optimizer(model, cfg)
scaler = torch.cuda.amp.GradScaler(enabled=cfg.AMP)
scheduler = None                                                     # legacy: none

class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {k: v.detach().clone().float() for k, v in model.state_dict().items()
                       if v.dtype.is_floating_point}
        self.backup = None
    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if k in self.shadow:
                self.shadow[k].mul_(self.decay).add_(v.detach().float(), alpha=1 - self.decay)
    def apply_to(self, model):
        self.backup = {k: v.detach().clone() for k, v in model.state_dict().items() if k in self.shadow}
        model.load_state_dict({k: (self.shadow[k].to(v.dtype) if k in self.shadow else v)
                               for k, v in model.state_dict().items()}, strict=True)
    def restore(self, model):
        if self.backup is None: return
        sd = model.state_dict(); sd.update(self.backup)
        model.load_state_dict(sd, strict=True); self.backup = None

ema = EMA(model, cfg.EMA_DECAY) if cfg.USE_EMA else None

n_all = sum(p.numel() for p in model.parameters())
n_new = sum(p.numel() for n, p in model.named_parameters() if n.split(".")[0] in NEW_PREFIX)
with torch.no_grad():
    probe = forward_logits(model, torch.zeros(1, 3, cfg.IMG_SIZE, cfg.IMG_SIZE, device=DEVICE))
assert probe.shape[-2:] == (cfg.IMG_SIZE, cfg.IMG_SIZE), probe.shape
assert probe.shape[1] == cfg.N_CLASSES, probe.shape
print(f"params: {n_all/1e6:.3f} M | new modules {n_new/1e6:.3f} M "
      f"(+{100*n_new/max(n_all-n_new,1):.2f}% over the B0 base)")
print(f"logits after upsample: {tuple(probe.shape)}")
print(f"param groups: {len(optimizer.param_groups)} | lr mode: {cfg.LR_MODE} | "
      f"lrs: {[g['lr'] for g in optimizer.param_groups]} | "
      f"wd: {optimizer.param_groups[0]['weight_decay']} | scheduler: {cfg.SCHEDULER} | "
      f"clip: {cfg.CLIP_NORM} | EMA: {bool(ema)}")
print(f"drop_path_rate={model.encoder.config.drop_path_rate} "
      f"classifier_dropout={model.encoder.config.classifier_dropout_prob} (checkpoint defaults)")

## 13. Metrics — the legacy estimator and the pooled one, side by side

`legacy_batch_miou` is a literal transcription of the legacy `mean_iou`: IoU is formed **within one
batch of 16 images**, classes whose batch union is zero are dropped, and the epoch score is the
unweighted mean over batches. That is a mean of ratios; the confusion matrix gives a ratio of means.
Different estimators of the same quantity, and they do not agree — a batch holding fifty River pixels
the model misses contributes a hard 0.0 that pulls the average down, while pooling barely notices
those pixels. Ignored pixels are dropped from both before anything is counted; in `ConfMat` that
filter must come **first**, because a label of 255 would otherwise overflow the `bincount` index.

Four series are logged every epoch:

| key | what it is |
|---|---|
| `val_miou_legacy` | the legacy estimator: batch-averaged, over the scored classes |
| `val_miou_global` | pooled from the confusion matrix, over the scored classes |
| `val_miou_core6` | pooled, 6 real classes — **the number comparable to your benchmark table** |
| `val_miou_core_region` | pooled, 6 real classes, restricted to CSA-core pixels (`relw >= CORE_THRESH`) |

Under `IGNORE_UNLABELLED = True` the second and third coincide, because the scored classes *are* the
six real ones; the first still differs, and that difference is the estimator alone. The fourth is
TC-specific and is the full/core protocol from the original notebook: the gap between it and
`val_miou_core6` separates model error from annotation error. With CSA off it is identical to
`val_miou_core6` by construction, since every weight is 1.0.

**`best.pt` is selected on `cfg.SELECTION_METRIC`, default `val_miou_core6`** — the same default the
B0 notebook uses, so both sides of the comparison keep the same epoch-selection rule. The legacy
script selected on the batch estimator, which is the noisiest of the four and not the one you report.

Per-class producer's (recall) and user's (precision) accuracy are reported separately, and this
matters: F1 here is algebraically `2*IoU/(1+IoU)` and carries no extra information, whereas PA and UA
tell you whether River at 0.14 is under- or over-prediction. Those need opposite remedies.

In [ ]:
from skimage.morphology import skeletonize

def legacy_batch_miou(logits, targets):
    preds = logits.argmax(dim=1)
    k = targets != cfg.IGNORE_INDEX          # inert in the 7-class scheme
    preds, targets = preds[k], targets[k]
    ious = []
    for c in range(cfg.N_CLASSES):
        inter = ((preds == c) & (targets == c)).sum().float()
        union = ((preds == c) | (targets == c)).sum().float()
        if union > 0:
            ious.append(inter / union)
    if len(ious) == 0:
        return 0.0
    return float(torch.mean(torch.stack(ious)))

class ConfMat:
    def __init__(self, n):
        self.n = n
        self.m = torch.zeros(n, n, dtype=torch.int64, device=DEVICE)
    def update(self, pred, tgt, mask=None):
        k = tgt != cfg.IGNORE_INDEX          # MUST come first: 255 would overflow the bincount
        if mask is not None:
            k = k & mask                     # CSA core region
        if k.sum() == 0: return
        self.m += torch.bincount(self.n * tgt[k].long() + pred[k].long(),
                                 minlength=self.n ** 2).reshape(self.n, self.n)
    def numpy(self): return self.m.cpu().numpy().astype(np.float64)

def metrics_from_cm(cm, names):
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; tot = cm.sum()
    iou = tp / np.maximum(tp + fp + fn, 1e-12)
    f1  = 2 * tp / np.maximum(2 * tp + fp + fn, 1e-12)
    prod = tp / np.maximum(cm.sum(1), 1e-12)      # recall / producer's accuracy
    user = tp / np.maximum(cm.sum(0), 1e-12)      # precision / user's accuracy
    freq = cm.sum(1) / max(tot, 1e-12)
    po = tp.sum() / max(tot, 1e-12)
    pe = (cm.sum(0) * cm.sum(1)).sum() / max(tot ** 2, 1e-12)
    return {
        "mIoU": float(np.nanmean(iou)), "mF1": float(np.nanmean(f1)),
        "mIoU_core6": float(np.nanmean(iou[CORE_CLASSES])),
        "mF1_core6": float(np.nanmean(f1[CORE_CLASSES])),
        "OA": float(po), "FWIoU": float((freq * iou).sum()),
        "kappa": float((po - pe) / max(1 - pe, 1e-12)),
        "per_class": {n: {"IoU": float(iou[i]), "F1": float(f1[i]),
                          "producer_acc": float(prod[i]), "user_acc": float(user[i]),
                          "support_px": int(cm.sum(1)[i])} for i, n in enumerate(names)},
        "confusion_matrix_raw": cm.astype(np.int64).tolist(),
        "confusion_matrix_norm": (cm / np.maximum(cm.sum(1, keepdims=True), 1e-12)).tolist(),
    }

def cldice_pair(pred_bin, gt_bin):
    if gt_bin.sum() == 0 and pred_bin.sum() == 0: return None
    sp = skeletonize(pred_bin.astype(bool)); sg = skeletonize(gt_bin.astype(bool))
    tprec = (sp & gt_bin.astype(bool)).sum() / max(sp.sum(), 1)
    tsens = (sg & pred_bin.astype(bool)).sum() / max(sg.sum(), 1)
    if tprec + tsens == 0: return 0.0
    return float(2 * tprec * tsens / (tprec + tsens))

print(f"metrics ready | {cfg.N_CLASSES} scored classes | selection metric: {cfg.SELECTION_METRIC}")

## 14. Efficiency metrics

Params, GFLOPs at 512x512 via `ptflops`, peak training VRAM, wall-clock, and inference FPS at batch 1
with 20 warm-up iterations.

**`ptflops`, matching the B0 notebook, deliberately.** The existing results carry a blocking
discrepancy: TC-SegFormer was measured at 12.27 GFLOPs against SegFormer-B0 at 15.71, even though TC
strictly contains B0 and adds a branch. That is two different counters, not two different models —
`fvcore` and `ptflops` disagree about SegFormer's functional attention matmuls. Counting both rows of
this comparison with one tool is what makes the pair quotable, so if you re-measure anything,
re-measure both sides.

`ptflops` hooks `nn.Module`s and misses functional matmuls, so transformer rows are under-counted
against CNN rows. Say so in the caption, or re-measure the whole table with one tool.

In [ ]:
class _FlopsWrapper(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return forward_logits(self.m, x)

def efficiency_report(m):
    rep = {"params_M": sum(p.numel() for p in m.parameters()) / 1e6,
           "params_new_M": sum(p.numel() for n, p in m.named_parameters()
                               if n.split(".")[0] in NEW_PREFIX) / 1e6}
    was_training = m.training
    m.eval()
    try:
        from ptflops import get_model_complexity_info
        with torch.cuda.amp.autocast(enabled=False):
            macs, _ = get_model_complexity_info(
                _FlopsWrapper(m), (3, cfg.IMG_SIZE, cfg.IMG_SIZE),
                as_strings=False, print_per_layer_stat=False, verbose=False)
        rep["GFLOPs_512"] = macs * 2 / 1e9
        rep["flops_counter"] = "ptflops"
    except Exception as e:
        rep["GFLOPs_512"] = None; rep["flops_error"] = str(e)[:160]
    x = torch.randn(1, 3, cfg.IMG_SIZE, cfg.IMG_SIZE, device=DEVICE)
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=cfg.AMP):
        for _ in range(20): forward_logits(m, x)
        torch.cuda.synchronize(); t0 = time.time()
        for _ in range(50): forward_logits(m, x)
        torch.cuda.synchronize()
    rep["inference_fps_bs1"] = 50 / (time.time() - t0)
    if was_training: m.train()
    return rep

EFF = efficiency_report(model)
print(json.dumps({k: (round(v, 4) if isinstance(v, float) else v) for k, v in EFF.items()}, indent=1))

## 15. Hugging Face persistence and resume

One repo per run — `geonusaf-tcsegformer-legacy-{c6|c7}-{recipe}-{mode}-fold{k}` — so nothing
overwrites anything and §21 can pull them all back. The prefix carries `legacy` deliberately: these
checkpoints must never be confused with the protocol runs when the results table is assembled, the
class scheme is in the name because a 6-way head cannot load a 7-way `last.pt`, and the recipe is in
the name because `legacy_pure` and `tc_full` are different experiments.

- `best.pt` — weights + config + metrics + `n_classes`, pushed whenever the selection metric improves.
- `last.pt` — full state including optimizer and scaler, pushed every `PUSH_LAST_EVERY` epochs.
  300 epochs will not fit one Kaggle session; this is what makes the fold resumable.

**Two loader details that have already cost a run each.** The B0 checkpoints store weights under
`model_state` while the original TC ones store them under `model`; `_find_state` detects the tensor
dict by content rather than matching a fixed key, so either loads. And the **architecture guard**
from the original TC notebook is kept: one repo per run means a checkpoint written by an older module
layout is still sitting there after any change, and `load_state_dict` will fail on it. Every
checkpoint carries `arch`, `recipe` and `n_classes`, and resume checks all three *and* the full
key/shape signature **before** touching the model, because a partially applied `load_state_dict`
leaves the network in a mixed state. On mismatch the run starts fresh and deletes the stale
artifacts. Bump `ARCH_VERSION` whenever you change the module layout.

The model card records the variant-leakage percentage from §8, the class scheme and the selection
metric. A card reporting a high mIoU without those next to it is a card that will eventually mislead
someone — most likely you, six weeks from now.

In [ ]:
from huggingface_hub import HfApi, create_repo, hf_hub_download
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN"); print("secret fallback:", e)

api = HfApi(token=HF_TOKEN) if HF_TOKEN else None
if api and cfg.HF_PUSH:
    create_repo(HF_REPO, token=HF_TOKEN, exist_ok=True, private=False)

def hf_push(local_path, name, msg="update"):
    if not (api and cfg.HF_PUSH): return
    try:
        api.upload_file(path_or_fileobj=str(local_path), path_in_repo=name,
                        repo_id=HF_REPO, token=HF_TOKEN, commit_message=msg)
    except Exception as e:
        print("HF upload failed:", str(e)[:160])

def hf_try_download(name, repo=None):
    if not api: return None
    try: return hf_hub_download(repo or HF_REPO, name, token=HF_TOKEN)
    except Exception: return None

def _find_state(ck):
    # B0 checkpoints use "model_state", the original TC ones use "model" -- detect by content
    for k in ("model_state", "model", "state_dict"):
        v = ck.get(k)
        if isinstance(v, dict) and v and all(torch.is_tensor(t) for t in v.values()):
            return v
    for v in ck.values():
        if isinstance(v, dict) and v and all(torch.is_tensor(t) for t in v.values()):
            return v
    return None

def _head_width(sd):
    for k in ("head.classifier.weight", "refine.weight", "decode_head.classifier.weight"):
        if k in sd: return sd[k].shape[0]
    return None

LEAK_PCT = len(leaky) / max(len(va_recs), 1) * 100

def push_model_card(best_metrics, epoch, legacy_miou, core_metrics=None):
    pc = best_metrics.get("per_class", {})
    L = ["---",
         "tags: [semantic-segmentation, remote-sensing, segformer, tc-segformer, geonusaf, legacy-recipe]",
         "library_name: transformers", "---", "",
         f"# GeoNUSAF - TC-SegFormer LEGACY RECIPE ({cfg.RECIPE}, {SCHEME_TAG}) - "
         f"{cfg.SPLIT_MODE}, fold {cfg.FOLD}", "",
         "TC-SegFormer trained with the pre-protocol B0 recipe (flat AdamW 3e-5, 300 epochs, legacy",
         "inverse-frequency weights, no scheduler/clipping/EMA). Read the split, class-scheme and",
         "metric fields below before putting any of these numbers next to a protocol run.", "",
         "| field | value |", "|---|---|",
         f"| architecture | TC-SegFormer (residual H/2 detail path) on {cfg.HF_MODEL_ID} |",
         f"| arch version | {cfg.ARCH_VERSION} |",
         f"| recipe | {cfg.RECIPE} (detail {cfg.USE_DETAIL}, csa {CSA_ENABLE}, sampler "
         f"{BALANCED_SAMPLER}, lovasz {LOVASZ_ENABLE}, cldice {CLDICE_ENABLE}) |",
         f"| split mode | {cfg.SPLIT_MODE} ({cfg.N_FOLDS}-fold, seed {cfg.KFOLD_SEED}) |",
         f"| fold | {cfg.FOLD} |",
         f"| VARIANT LEAKAGE | {len(leaky)}/{len(va_recs)} val images ({LEAK_PCT:.1f}%) have a mirror in train |",
         f"| classes | {cfg.N_CLASSES} ({', '.join(cfg.CLASS_NAMES)}) |",
         f"| unlabelled pixels | {'ignored (index 255)' if cfg.IGNORE_UNLABELLED else 'trained as class 6'} |",
         f"| class weights | {cfg.CLASS_WEIGHT_MODE}, counted over {cfg.WEIGHT_SOURCE}, cap {cfg.WEIGHT_CAP} |",
         f"| CSA band | tau={list(cfg.CSA_TAU)[:cfg.N_CLASSES]}, w_min={cfg.CSA_WMIN}, "
         f"cap {cfg.CSA_DMAX_M} m @ {cfg.TILE_METRES} m tile = {DMAX_PX:.2f} px |",
         f"| optimizer | AdamW lr {cfg.LR} flat, mode {cfg.LR_MODE} (new x{cfg.LR_NEW_MULT}), wd {cfg.WEIGHT_DECAY} |",
         f"| scheduler / clip / EMA | {cfg.SCHEDULER} / {cfg.CLIP_NORM} / {cfg.USE_EMA} |",
         f"| epochs / patience / batch | {cfg.MAX_EPOCHS} / {cfg.PATIENCE} / {cfg.BATCH_SIZE} |",
         f"| val transform | {'train transform (legacy)' if cfg.VAL_USES_TRAIN_TF else 'clean'} |",
         f"| selection metric | {cfg.SELECTION_METRIC} |",
         f"| best epoch | {epoch} |",
         f"| val mIoU (batch-averaged estimator) | {legacy_miou:.4f} |",
         f"| val mIoU (pooled, all scored classes) | {best_metrics.get('mIoU', float('nan')):.4f} |",
         f"| val mIoU (pooled, 6 real classes) | {best_metrics.get('mIoU_core6', float('nan')):.4f} |",
         f"| val mIoU (CSA core region, 6 classes) | "
         f"{(core_metrics or {}).get('mIoU_core6', float('nan')):.4f} |",
         f"| val OA | {best_metrics.get('OA', float('nan')):.4f} |",
         f"| val kappa | {best_metrics.get('kappa', float('nan')):.4f} |", "",
         "## Per-class (validation, pooled)", "| class | IoU | F1 | PA (recall) | UA (precision) |",
         "|---|---|---|---|---|"]
    for k, v in pc.items():
        L.append(f"| {k} | {v['IoU']:.4f} | {v['F1']:.4f} | "
                 f"{v['producer_acc']:.4f} | {v['user_acc']:.4f} |")
    p = f"{cfg.OUT_DIR}/README.md"; open(p, "w").write("\n".join(L))
    hf_push(p, "README.md", f"card epoch {epoch}")

print("HF ready:", HF_REPO if api else "(disabled -- no token)",
      f"| scheme {SCHEME_TAG} | leakage on this fold: {LEAK_PCT:.1f}%")

## 16. Weights & Biases

Every log passes `step=epoch`, so the x-axis means what it says. The legacy script called `wandb.log`
once per epoch **and** again inside `log_images_wandb` every 5 epochs with no `step=` argument; W&B
advances its own counter on each call, so 300 epochs rendered as ~360 steps and the axis silently
stopped being the epoch number. That is the whole explanation for curves that appear to run past
their epoch budget.

All four mIoU definitions are logged, plus the loss components, both grad-norm series, the detail
gate, and `overfit_gap`.

In [ ]:
WB = None
try:
    import wandb
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    WB = wandb.init(project=cfg.WANDB_PROJECT, name=RUN_NAME, id=RUN_NAME.replace("/", "-"),
                    resume="allow",
                    config={**asdict(cfg), **EFF, "variant_leakage_pct": LEAK_PCT,
                            "scheme": SCHEME_TAG,
                            "train_images": len(tr_recs), "val_images": len(va_recs),
                            "csa_enable": CSA_ENABLE, "balanced_sampler": BALANCED_SAMPLER,
                            "lovasz_enable": LOVASZ_ENABLE, "cldice_enable": CLDICE_ENABLE})
    print("wandb:", WB.url)
except Exception as e:
    print("wandb disabled:", str(e)[:160]); WB = None

## 17. Train / validate loops

The update is the legacy one: forward under `autocast`, weighted CE, `scaler.scale(loss).backward()`,
`scaler.step`, `scaler.update()`. No `unscale_`, no clipping, no scheduler step — unless you set
`CLIP_NORM`, in which case `unscale_` runs first, because clipping loss-scaled gradients would make
the threshold meaningless and is exactly the bug that lets 6e5 norms through.

The only structural change from the B0 loop is that the loss is computed **outside** `autocast` in
fp32, which is required by Lovasz and is arithmetically identical for CE.

### The grad-norm curve
The legacy script computed its gradient norm straight after `scaler.scale(loss).backward()`, over
gradients still multiplied by the loss scale — typically 2^16. That is why those plots read 100,000
to 600,000 rather than 1 to 10, and why an apparent explosion never destabilised anything. Both
series are logged: `grad_norm_scaled` reproduces the old curve so the two runs overlay, and
`grad_norm_true` divides by the current loss scale, which is the real number. Dividing is exact, so
this costs nothing and does not touch the update. **`grad_norm_true` is the number to watch under
`tc_full`** — see the note in §9.

Validation runs over the val loader (clean by default, per §10) and accumulates the legacy per-batch
estimator, a pooled confusion matrix, and a second pooled matrix over the CSA core region, all in one
pass.

In [ ]:
def train_one_epoch(epoch):
    model.train()
    criterion.set_epoch(epoch)
    tot, n = 0.0, 0
    agg = dict(ce=0.0, lovasz=0.0, cldice=0.0); nb = 0
    gn_scaled, gn_true = [], []
    for it, (x, y, w) in enumerate(train_ld):
        x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
        w = w.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=cfg.AMP):
            logits = forward_logits(model, x)
        loss, parts = criterion(logits, y, w)                  # fp32, outside autocast

        scaler.scale(loss).backward()

        # legacy grad-norm probe: computed on SCALED grads, exactly as the legacy script did
        sq = 0.0
        for p in model.parameters():
            if p.grad is not None:
                sq += float(p.grad.data.norm(2)) ** 2
        gs = sq ** 0.5
        gn_scaled.append(gs)
        gn_true.append(gs / max(scaler.get_scale(), 1e-12) if cfg.AMP else gs)

        if cfg.CLIP_NORM:                                      # legacy: None -> block skipped
            scaler.unscale_(optimizer)                         # MUST precede clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.CLIP_NORM)

        scaler.step(optimizer)
        scaler.update()
        if ema: ema.update(model)

        tot += float(loss.detach()) * x.size(0); n += x.size(0)
        agg["ce"] += parts["ce"]; agg["lovasz"] += parts["lovasz"]; agg["cldice"] += parts["cldice"]
        nb += 1
        if it % 20 == 0:
            print(f"  e{epoch} it{it}/{len(train_ld)} loss {float(loss.detach()):.4f} "
                  f"ce {parts['ce']:.4f} gnorm_true {gn_true[-1]:.3f} (scaled {gs:.0f})", flush=True)
    k = max(nb, 1)
    return {"train_loss": tot / max(n, 1), "train_ce": agg["ce"] / k,
            "train_lovasz": agg["lovasz"] / k, "train_cldice": agg["cldice"] / k,
            "lovasz_lambda": criterion.lam, "cldice_mu": criterion.mu,
            "grad_norm_scaled": float(np.mean(gn_scaled)),
            "grad_norm_true": float(np.mean(gn_true)),
            "grad_norm_true_max": float(np.max(gn_true)),
            "gate": float(model.gate.detach()) if cfg.USE_DETAIL else 0.0}

@torch.no_grad()
def validate(loader=None, m=None):
    loader = loader if loader is not None else val_ld
    m = m if m is not None else model
    if ema and m is model: ema.apply_to(m)
    m.eval()
    cm, cm_core = ConfMat(cfg.N_CLASSES), ConfMat(cfg.N_CLASSES)
    tot, nce, n, mious = 0.0, 0.0, 0, []
    for x, y, w in loader:
        x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
        w = w.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=cfg.AMP):
            logits = forward_logits(m, x)
        loss, parts = criterion(logits, y, w)
        lf = logits.float()
        pred = lf.argmax(1)
        mious.append(legacy_batch_miou(lf, y))          # legacy estimator, per batch
        cm.update(pred, y)                              # pooled estimator
        cm_core.update(pred, y, mask=(w >= cfg.CORE_THRESH))
        tot += float(loss.detach()) * x.size(0); nce += parts["ce"] * x.size(0); n += x.size(0)
    if ema and m is model: ema.restore(m)
    m.train()
    return ({"val_loss": tot / max(n, 1), "val_ce": nce / max(n, 1)},
            float(np.mean(mious)),
            metrics_from_cm(cm.numpy(), cfg.CLASS_NAMES),
            metrics_from_cm(cm_core.numpy(), cfg.CLASS_NAMES))

print("loops ready")

## 18. Main loop with automatic resume

Early stopping is the legacy logic transcribed, including its shape: the patience counter resets on
any improvement in `cfg.SELECTION_METRIC`, and the `break` lives in the `else` branch. There is no
`min_delta` — the original TC notebook added one at 0.002, the legacy script has none, and this
notebook follows the legacy script and the B0 rebuild.

**Budget.** ~980 train images at batch 16 is ~62 steps/epoch. The per-step gradient-norm probe walks
every parameter tensor and costs real time; it is kept because it is what produced the comparison
curves, but it is the first thing to delete if you are short on session hours. TC is ~15% more FLOPs
than B0 and the Lovasz term adds an fp32 sort, so expect roughly 60-95 s/epoch under `tc_full` and
closer to the B0 figure under `legacy_pure`. **300 epochs is 5-8 h per fold and will not finish in
one 12 h session together with the repack.** Plan on two sessions per fold; rerun the notebook
unchanged and it continues from `last.pt`. RNG state is not restored, so a resumed fold is
reproducible only up to the resume point.

**Sanity gates.** Because the refinement is initialised near zero, epoch 0 is effectively plain
SegFormer-B0, so this run should track the B0 curve from the start and then diverge upward. If it
tracks *below* it early, the new modules are fighting the pretrained head — lower `LR_NEW_MULT`. And
watch `gate`: if it has not moved from 0.1 by epoch 30, the detail path is inert.

In [ ]:
def _compatible(sd):
    cur = model.state_dict()
    if set(cur) != set(sd):
        return False, (f"key sets differ (checkpoint-only {sorted(set(sd)-set(cur))[:4]}, "
                       f"model-only {sorted(set(cur)-set(sd))[:4]})")
    for k in cur:
        if tuple(cur[k].shape) != tuple(sd[k].shape):
            return False, f"shape mismatch at {k}: {tuple(sd[k].shape)} vs {tuple(cur[k].shape)}"
    return True, ""

def _purge_stale(reason):
    print(f"!! IGNORING the remote checkpoint -- {reason}")
    print(f"   This run starts from scratch; stale artifacts are deleted from {HF_REPO}")
    if api and cfg.HF_PUSH:
        for f in ("last.pt", "best.pt"):
            try: api.delete_file(path_in_repo=f, repo_id=HF_REPO)
            except Exception: pass

start_epoch, best_miou, best_epoch, bad, history = 0, 0.0, -1, 0, []
best_metrics, best_core, best_legacy = {}, {}, 0.0

ck_path = hf_try_download("last.pt") if cfg.RESUME else None
if ck_path:
    st = torch.load(ck_path, map_location=DEVICE, weights_only=False)
    sd = _find_state(st)
    nc = st.get("n_classes", _head_width(sd) if sd is not None else None)
    ok, why = _compatible(sd) if sd is not None else (False, "no tensor state dict in checkpoint")
    if nc is not None and nc != cfg.N_CLASSES:
        _purge_stale(f"it has a {nc}-class head, this run has {cfg.N_CLASSES} "
                     f"(check HF_REPO -- it should contain -{SCHEME_TAG}-)")
    elif st.get("arch") != cfg.ARCH_VERSION:
        _purge_stale(f"architecture {st.get('arch', '<pre-versioning>')!r} != {cfg.ARCH_VERSION!r}")
    elif st.get("recipe") not in (None, cfg.RECIPE):
        _purge_stale(f"recipe {st.get('recipe')!r} != {cfg.RECIPE!r}")
    elif not ok:
        _purge_stale(why)
    else:
        model.load_state_dict(sd); optimizer.load_state_dict(st["optim_state"])
        scaler.load_state_dict(st["scaler_state"])
        if ema and st.get("ema"): ema.shadow = {k: v.to(DEVICE) for k, v in st["ema"].items()}
        start_epoch = st["epoch"] + 1; best_miou = st["best_miou"]; best_epoch = st["best_epoch"]
        bad = st["bad"]; history = st["history"]
        print(f"RESUMED from epoch {st['epoch']} "
              f"(best {cfg.SELECTION_METRIC} {best_miou:.4f} @ e{best_epoch})")
elif cfg.RESUME:
    print("no remote checkpoint -- starting fresh")
else:
    print("RESUME=False -- starting fresh, remote checkpoint ignored")

torch.cuda.reset_peak_memory_stats()
t_start = time.time()

for epoch in range(start_epoch, cfg.MAX_EPOCHS):
    te = time.time()
    tr = train_one_epoch(epoch)
    vl, va_legacy, m, mcore = validate()
    peak_gb = torch.cuda.max_memory_allocated() / 1e9

    row = {"epoch": epoch, **tr, **vl,
           "overfit_gap": vl["val_loss"] - tr["train_loss"],
           "gen_gap_ce": vl["val_ce"] - tr["train_ce"],
           "val_miou_legacy": va_legacy,                  # the legacy batch-averaged estimator
           "val_miou_global": m["mIoU"],                  # pooled, all scored classes
           "val_miou_core6": m["mIoU_core6"],             # pooled, 6 real classes
           "val_miou_core_region": mcore["mIoU_core6"],   # pooled, CSA core pixels, 6 real classes
           "val_mF1": m["mF1"], "val_OA": m["OA"], "val_kappa": m["kappa"],
           "val_FWIoU": m["FWIoU"],
           "lr": optimizer.param_groups[0]["lr"], "peak_vram_gb": peak_gb,
           "epoch_sec": time.time() - te,
           **{f"IoU/{k}": v["IoU"] for k, v in m["per_class"].items()}}
    history.append(row)
    if WB: WB.log(row, step=epoch)
    print(f"[e{epoch}] CE {tr['train_ce']:.4f}/{vl['val_ce']:.4f} (gap {row['gen_gap_ce']:+.2f}) "
          f"lam {tr['lovasz_lambda']:.2f} mu {tr['cldice_mu']:.2f} | "
          f"mIoU legacy {va_legacy:.4f} / pooled {m['mIoU']:.4f} / core6 {m['mIoU_core6']:.4f} "
          f"(csa-core {mcore['mIoU_core6']:.4f}) | "
          f"road {m['per_class']['Road']['IoU']:.3f} river {m['per_class']['River']['IoU']:.3f} | "
          f"gate {tr['gate']:.3f} | gnorm {tr['grad_norm_true']:.2f} | "
          f"{row['epoch_sec']:.0f}s | vram {peak_gb:.1f}GB")
    if epoch == 30 and cfg.USE_DETAIL and abs(tr["gate"] - cfg.DETAIL_GATE_INIT) < 0.005:
        print("   ^ the detail gate has not moved from its init in 30 epochs -- the detail path is "
              "inert and this run is B0 with extra FLOPs. Check LR_MODE / LR_NEW_MULT.")
    if tr["grad_norm_true"] > 50 and epoch > 5:
        print("   ^ grad_norm_true above 50: if this persists while train_ce stalls, set "
              "CLIP_NORM=1.0 and restart the fold (see section 9).")

    sel = row[cfg.SELECTION_METRIC]
    if sel > best_miou:
        best_miou, best_epoch, bad = sel, epoch, 0
        best_metrics, best_core, best_legacy = m, mcore, va_legacy
        sd = {k: v.detach().clone() for k, v in model.state_dict().items()}
        if ema:
            for k, v in ema.shadow.items(): sd[k] = v.to(sd[k].dtype)
        p = f"{cfg.OUT_DIR}/best.pt"
        torch.save({"model_state": {k: v.cpu() for k, v in sd.items()},
                    "cfg": asdict(cfg), "arch": cfg.ARCH_VERSION, "recipe": cfg.RECIPE,
                    "n_classes": cfg.N_CLASSES, "class_names": list(cfg.CLASS_NAMES),
                    "metrics": m, "metrics_core": mcore, "legacy_miou": va_legacy,
                    "efficiency": EFF, "epoch": epoch}, p)
        hf_push(p, "best.pt", f"best {cfg.SELECTION_METRIC} {best_miou:.4f} @ e{epoch}")
        push_model_card(best_metrics, epoch, best_legacy, best_core)
        print(f"  -> new best {best_miou:.4f}, pushed")
    else:
        bad += 1

    if (epoch % cfg.PUSH_LAST_EVERY == 0) or (bad >= cfg.PATIENCE) or (epoch == cfg.MAX_EPOCHS - 1):
        p = f"{cfg.OUT_DIR}/last.pt"
        torch.save({"model_state": model.state_dict(), "optim_state": optimizer.state_dict(),
                    "scaler_state": scaler.state_dict(), "ema": (ema.shadow if ema else None),
                    "epoch": epoch, "best_miou": best_miou, "best_epoch": best_epoch,
                    "bad": bad, "history": history, "cfg": asdict(cfg),
                    "arch": cfg.ARCH_VERSION, "recipe": cfg.RECIPE,
                    "n_classes": cfg.N_CLASSES}, p)
        hf_push(p, "last.pt", f"state e{epoch}")

    if bad >= cfg.PATIENCE:
        print(f"EARLY STOP at epoch {epoch} (best {best_miou:.4f} @ e{best_epoch})"); break

wall_h = (time.time() - t_start) / 3600
print(f"training done | best {cfg.SELECTION_METRIC} {best_miou:.4f} @ epoch {best_epoch} | "
      f"this session {wall_h:.2f} h")

## 19. Final evaluation of the best checkpoint — three protocols

The best checkpoint is scored three ways, and the differences between them are the point:

1. **Legacy protocol** — through `val_ld`, using the per-batch estimator. Comparable to the legacy
   figure (and identical to the clean pass when `VAL_USES_TRAIN_TF` is `False`, apart from the
   estimator).
2. **Clean protocol** — through `clean_ld` (resize only, deterministic), pooled from a confusion
   matrix, reported over all scored classes and over the 6 real ones. Comparable to the benchmark
   table.
3. **CSA core protocol** — the clean pass restricted to `relw >= CORE_THRESH`, i.e. pixels outside
   the unreliable boundary band. The gap separates model error from annotation error.

clDice for Road and River and the efficiency block run on the clean pass only, and both are computed
with ignored pixels excluded, so annotation gaps cannot contribute a phantom road. clDice averages
over images where the class is present in ground truth or prediction; images where it is absent from
both are skipped rather than scored 1.0, which would inflate River badly. The `n` behind each clDice
figure is recorded, because the denominators differ between runs and a clDice quoted without its
count is not comparable.

**On the core protocol and reviewers.** "We evaluate on eroded interiors and our numbers go up" is
circular on its own. The defence is a small independently re-annotated gold subset (30-50 tiles,
tight boundaries, ideally a second annotator) reported alongside, which also yields an
inter-annotator agreement figure — arguably the most useful single number the paper could carry.

In [ ]:
# On a resumed session best.pt may exist only on the Hub: this session may never have improved.
_bp = f"{cfg.OUT_DIR}/best.pt"
if not os.path.exists(_bp):
    _bp = hf_try_download("best.pt")
    assert _bp, "no best.pt locally or on the Hub -- did any epoch ever improve?"
    print("best.pt pulled from the Hub (no improvement in this session)")
_ck = torch.load(_bp, map_location=DEVICE, weights_only=False)
_sd = _find_state(_ck)
_nc = _ck.get("n_classes", _head_width(_sd))
assert _nc == cfg.N_CLASSES, (
    f"best.pt has a {_nc}-class head but this run is {cfg.N_CLASSES}-class -- wrong repo")
best_model = build_model().to(DEVICE)
best_model.load_state_dict(_sd)
best_model.eval()

# ---- 1. legacy protocol: val loader + per-batch estimator ----
_, legacy_miou_final, legacy_pooled, _ = validate(loader=val_ld, m=best_model)

# ---- 2/3. clean protocol: deterministic loader, pooled + core confusion matrices, clDice ----
cm = ConfMat(cfg.N_CLASSES); cm_core = ConfMat(cfg.N_CLASSES)
cld = {1: [], 2: []}                                   # Road, River
per_image = []
with torch.no_grad():
    for bi, (x, y, w) in enumerate(clean_ld):
        x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
        w = w.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=cfg.AMP):
            logits = forward_logits(best_model, x)
        pred = logits.float().argmax(1)
        cm.update(pred, y)
        cm_core.update(pred, y, mask=(w >= cfg.CORE_THRESH))
        pn, yn = pred.cpu().numpy(), y.cpu().numpy()
        valid = yn != VOID_LABEL                       # do not score clDice inside annotation gaps
        for b in range(pn.shape[0]):
            i = bi * cfg.BATCH_SIZE + b
            c1 = ConfMat(cfg.N_CLASSES); c1.update(pred[b], y[b])
            m1 = c1.numpy()
            tp = np.diag(m1); un = m1.sum(0) + m1.sum(1) - tp
            sel = [k for k in CORE_CLASSES if un[k] > 0]      # absent classes are not scored 0
            per_image.append({"idx": i, "name": va_recs[i][2],
                              "mIoU_core6": float(np.mean(tp[sel] / un[sel])) if sel else 0.0})
            for c in cld:
                v = cldice_pair((pn[b] == c) & valid[b], (yn[b] == c) & valid[b])
                if v is not None: cld[c].append(v)

final = metrics_from_cm(cm.numpy(), cfg.CLASS_NAMES)
final_core = metrics_from_cm(cm_core.numpy(), cfg.CLASS_NAMES)
final["clDice"] = {cfg.CLASS_NAMES[c]: (float(np.mean(v)) if v else None) for c, v in cld.items()}
final["clDice_n_images"] = {cfg.CLASS_NAMES[c]: len(v) for c, v in cld.items()}
final["miou_batch_estimator"] = legacy_miou_final
final["miou_pooled_valloader"] = legacy_pooled["mIoU"]
final["core_region"] = final_core
final["variant_leakage_pct"] = LEAK_PCT
final.update(efficiency_report(best_model))
final["peak_train_vram_gb"] = float(torch.cuda.max_memory_allocated() / 1e9)
final["train_wall_clock_h_this_session"] = wall_h
final["best_epoch"] = best_epoch
final["run"] = {"model": "TC-SegFormer (legacy recipe)", "recipe": cfg.RECIPE,
                "split_mode": cfg.SPLIT_MODE, "fold": cfg.FOLD, "seed": cfg.SEED,
                "n_classes": cfg.N_CLASSES, "ignore_unlabelled": cfg.IGNORE_UNLABELLED,
                "selection_metric": cfg.SELECTION_METRIC,
                "class_weight_mode": cfg.CLASS_WEIGHT_MODE,
                "csa": CSA_ENABLE, "sampler": BALANCED_SAMPLER,
                "lovasz": LOVASZ_ENABLE, "cldice": CLDICE_ENABLE, "lr_mode": cfg.LR_MODE}

NC = cfg.N_CLASSES
print("=" * 76)
print(f"batch-averaged estimator, val loader, {NC} classes : {legacy_miou_final:.4f}")
print(f"pooled,                   val loader, {NC} classes : {legacy_pooled['mIoU']:.4f}")
print(f"pooled,                   clean loader, {NC} cls   : {final['mIoU']:.4f}")
print(f"pooled,                   clean loader, 6 real cls : {final['mIoU_core6']:.4f}"
      f"   <- the number for the paper")
print(f"pooled,                   CSA core region, 6 cls   : {final_core['mIoU_core6']:.4f}"
      f"   (gap {final_core['mIoU_core6']-final['mIoU_core6']:+.4f} = annotation-noise share)")
print(f"variant leakage on this fold                       : {LEAK_PCT:.1f}%")
print("=" * 76)
print(f"OA {final['OA']:.4f} | FWIoU {final['FWIoU']:.4f} | kappa {final['kappa']:.4f} | "
      f"mF1(6) {final['mF1_core6']:.4f}")
print(f"clDice Road {final['clDice']['Road']} (n={final['clDice_n_images']['Road']}) | "
      f"River {final['clDice']['River']} (n={final['clDice_n_images']['River']})")
print("\nper class (clean, pooled):")
for k, v in final["per_class"].items():
    print(f"  {k:<14} IoU {v['IoU']:.4f}  F1 {v['F1']:.4f}  "
          f"PA {v['producer_acc']:.4f}  UA {v['user_acc']:.4f}  "
          f"coreIoU {final_core['per_class'][k]['IoU']:.4f}")

## 20. Persist results

`results_{recipe}_{mode}_fold{k}.json` (full, including both confusion matrices), a flat `.csv` row
for the comparison table, `history_*.csv` for the curves and `per_image_*.csv` for the qualitative
picks, all pushed to the run's repo.

The flat row uses **the same column names as the B0 notebook** — `miou_batch_estimator`,
`miou_pooled_valloader`, `mIoU_clean_allclasses`, `mIoU_clean_core6` — which is what lets §21 join the
two sides without a rename map. It also carries `recipe`, `n_classes`, `ignore_unlabelled`,
`selection_metric`, `val_transform`, `variant_leakage_pct`, `class_weight_mode` and `lr_mode`,
because the failure mode this whole exercise exists to prevent is a legacy number and a protocol
number ending up in the same table column with nothing to tell them apart.

In [ ]:
import csv
tag = f"{cfg.RECIPE}_{cfg.SPLIT_MODE}_fold{cfg.FOLD}"
jp = f"{cfg.OUT_DIR}/results_{tag}.json"
json.dump(final, open(jp, "w"), indent=2)

flat = {"model": "TC-SegFormer-legacy", "recipe": cfg.RECIPE, "split_mode": cfg.SPLIT_MODE,
        "fold": cfg.FOLD, "seed": cfg.SEED, "n_classes": cfg.N_CLASSES,
        "ignore_unlabelled": cfg.IGNORE_UNLABELLED, "selection_metric": cfg.SELECTION_METRIC,
        "val_transform": "train" if cfg.VAL_USES_TRAIN_TF else "clean",
        "variant_leakage_pct": LEAK_PCT, "class_weight_mode": cfg.CLASS_WEIGHT_MODE,
        "lr": cfg.LR, "lr_mode": cfg.LR_MODE, "epochs_max": cfg.MAX_EPOCHS,
        "best_epoch": final["best_epoch"],
        "csa": CSA_ENABLE, "sampler": BALANCED_SAMPLER,
        "lovasz": LOVASZ_ENABLE, "cldice": CLDICE_ENABLE,
        "miou_batch_estimator": final["miou_batch_estimator"],
        "miou_pooled_valloader": final["miou_pooled_valloader"],
        "mIoU_clean_allclasses": final["mIoU"], "mIoU_clean_core6": final["mIoU_core6"],
        "mIoU_csa_core_region": final_core["mIoU_core6"],
        "mF1_core6": final["mF1_core6"], "OA": final["OA"],
        "FWIoU": final["FWIoU"], "kappa": final["kappa"],
        "params_M": final["params_M"], "GFLOPs_512": final["GFLOPs_512"],
        "flops_counter": final.get("flops_counter"),
        "fps_bs1": final["inference_fps_bs1"], "peak_vram_gb": final["peak_train_vram_gb"],
        "train_hours_this_session": wall_h}
for k, v in final["per_class"].items():
    flat[f"IoU_{k}"] = v["IoU"]; flat[f"F1_{k}"] = v["F1"]
    flat[f"PA_{k}"] = v["producer_acc"]; flat[f"UA_{k}"] = v["user_acc"]
for k, v in final["clDice"].items():
    flat[f"clDice_{k}"] = v; flat[f"clDice_n_{k}"] = final["clDice_n_images"][k]

cp = f"{cfg.OUT_DIR}/results_{tag}.csv"
with open(cp, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(flat)); w.writeheader(); w.writerow(flat)

hp = f"{cfg.OUT_DIR}/history_{tag}.csv"
with open(hp, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(history[0])); w.writeheader(); w.writerows(history)

pip_ = f"{cfg.OUT_DIR}/per_image_{tag}.csv"
with open(pip_, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(per_image[0])); w.writeheader(); w.writerows(per_image)

for p, n in [(jp, f"results_{tag}.json"), (cp, f"results_{tag}.csv"),
             (hp, f"history_{tag}.csv"), (pip_, f"per_image_{tag}.csv")]:
    hf_push(p, n, "results")
if WB: WB.summary.update(flat); WB.finish()
print("results pushed to", HF_REPO)

## 21. Aggregation — and the TC-minus-B0 delta, which is the actual deliverable

Two blocks. The first pulls every TC-legacy fold and reports mean +/- SD for each mIoU definition per
recipe and split mode, exactly as the B0 notebook does.

The second is the one that matters. It pulls the **B0 legacy** result for the same split mode and
fold and prints the paired difference. This is the comparison the whole notebook exists to produce,
and it is readable even under `legacy_kfold`: both models are scored on the same leaky validation
set, so the leakage is common-mode and cancels in the difference. The absolute numbers do not survive
that split; the delta does.

Both sides use `candidate_repos`, which tries the `c6-` and `c7-` spellings and the untagged one, so
runs made before the scheme tag existed are still found. The join also checks `n_classes` on both
sides and refuses to pair a 6-class run with a 7-class one — that comparison is meaningless and it is
exactly the kind of thing that slips into a table unnoticed.

What to read:

- **`miou_batch_estimator` delta** — TC vs B0 under the exact conditions that produced the legacy
  figure. Honest as a paired comparison, not quotable as an absolute.
- **`mIoU_clean_core6` delta under `block`** — the only row that belongs in the paper unqualified.
- **The two deltas against each other** — if TC's advantage is much larger under `legacy_kfold` than
  under `block`, then what TC is adding is partly memorisation capacity rather than segmentation
  quality, and the detail path is the obvious suspect since it is the highest-resolution part of the
  model. Worth knowing before a reviewer asks.

A single-fold delta below ~0.01 mIoU is inside the fold-to-fold noise on 408 tiles. Run all folds
before concluding anything, and remember the existing block-fold SD on this dataset is 7-35x the
random-split SD.

In [ ]:
import pandas as pd
MIOU_COLS = ["miou_batch_estimator", "miou_pooled_valloader",
             "mIoU_clean_allclasses", "mIoU_clean_core6"]
MAX_FOLDS = 5
MODES = ["legacy_kfold", "random", "block"]

def candidate_repos(user, prefix, mid, mode, k):
    # "" covers runs made before the c6/c7 tag was introduced
    return [f"{user}/{prefix}-{t}{mid}{mode}-fold{k}" for t in ("c6-", "c7-", "")]

def pull(repos, fname):
    for repo in repos:
        try: return pd.read_csv(hf_hub_download(repo, fname, token=HF_TOKEN))
        except Exception: continue
    return None

def tc_repos(rec, mode, k):
    return candidate_repos(cfg.HF_USER, cfg.HF_PREFIX, f"{rec}-", mode, k)

def b0_repos(mode, k):
    return candidate_repos(cfg.B0_HF_USER, cfg.B0_HF_PREFIX, "", mode, k)

# ---- block 1: every TC-legacy run ----
rows = []
for rec in RECIPES:
    for mode in MODES:
        for k in range(MAX_FOLDS):
            r = pull(tc_repos(rec, mode, k), f"results_{rec}_{mode}_fold{k}.csv")
            if r is not None: rows.append(r)

if rows:
    df = pd.concat(rows, ignore_index=True)
    df.to_csv(f"{cfg.OUT_DIR}/aggregate_tcsegformer_legacy.csv", index=False)
    print("--- TC-SegFormer legacy: the four mIoU definitions ---")
    print(df.groupby(["recipe", "split_mode"])[MIOU_COLS].agg(["mean", "std"]).round(4).to_string())
    print("\n--- leakage present on each split mode ---")
    print(df.groupby("split_mode")["variant_leakage_pct"].mean().round(1).to_string())
    if df.n_classes.nunique() > 1:
        print("\nNOTE: these rows mix 6-class and 7-class runs -- check n_classes before "
              "comparing any two.")
else:
    df = None
    print("no TC fold CSVs found yet -- run the folds first")

# ---- block 2: the paired TC minus B0 delta ----
print("\n" + "=" * 76)
print("PAIRED DELTA vs the SegFormer-B0 legacy run (same split mode, same fold, same class scheme)")
print("=" * 76)
pairs, skipped = [], []
for rec in RECIPES:
    for mode in MODES:
        for k in range(MAX_FOLDS):
            tc = pull(tc_repos(rec, mode, k), f"results_{rec}_{mode}_fold{k}.csv")
            b0 = pull(b0_repos(mode, k), f"results_{mode}_fold{k}.csv")
            if tc is None or b0 is None: continue
            if int(tc.n_classes.iloc[0]) != int(b0.n_classes.iloc[0]):
                skipped.append(f"{rec}/{mode}/f{k}: TC is {int(tc.n_classes.iloc[0])}-class, "
                               f"B0 is {int(b0.n_classes.iloc[0])}-class")
                continue
            row = {"recipe": rec, "split_mode": mode, "fold": k,
                   "n_classes": int(tc.n_classes.iloc[0]),
                   "leak_pct": float(tc.variant_leakage_pct.iloc[0])}
            for c in MIOU_COLS:
                row[f"TC_{c}"] = float(tc[c].iloc[0])
                row[f"B0_{c}"] = float(b0[c].iloc[0])
                row[f"d_{c}"] = float(tc[c].iloc[0]) - float(b0[c].iloc[0])
            pairs.append(row)

if pairs:
    pdf = pd.DataFrame(pairs)
    pdf.to_csv(f"{cfg.OUT_DIR}/tc_vs_b0_legacy.csv", index=False)
    hf_push(f"{cfg.OUT_DIR}/tc_vs_b0_legacy.csv", "tc_vs_b0_legacy.csv", "paired delta")
    for _, r in pdf.iterrows():
        print(f"  {r.recipe:<12} {r.split_mode:<13} f{int(r.fold)} c{int(r.n_classes)} "
              f"(leak {r.leak_pct:4.1f}%): "
              f"batch-est {r.TC_miou_batch_estimator:.4f} vs {r.B0_miou_batch_estimator:.4f} "
              f"({r.d_miou_batch_estimator:+.4f})  |  "
              f"clean6 {r.TC_mIoU_clean_core6:.4f} vs {r.B0_mIoU_clean_core6:.4f} "
              f"({r.d_mIoU_clean_core6:+.4f})")
    print("\nmean delta per recipe x split mode:")
    print(pdf.groupby(["recipe", "split_mode"])[["d_miou_batch_estimator",
                                                 "d_mIoU_clean_core6"]].agg(["mean", "std", "count"])
          .round(4).to_string())
    print("\nReminder: absolute mIoU under legacy_kfold is not publishable (see section 8).")
    print("The delta is, because both models see the same leaky validation set.")
else:
    print("  no matching B0 legacy runs found. Check B0_HF_USER / B0_HF_PREFIX in section 2,")
    print("  and that the B0 fold you want to pair with has finished and pushed its CSV.")
for s in skipped:
    print("  SKIPPED (class-scheme mismatch)", s)

# ---- block 3: did each fold converge? ----
print("\n--- did each TC fold converge? ---")
for rec in RECIPES:
    for mode in MODES:
        for k in range(MAX_FOLDS):
            h = pull(tc_repos(rec, mode, k), f"history_{rec}_{mode}_fold{k}.csv")
            if h is None or "val_miou_core6" not in h: continue
            em, last = int(h.val_miou_core6.idxmax()), int(h.epoch.max())
            flag = "  <-- STILL IMPROVING AT THE END" if em >= last - 5 else ""
            print(f"  {rec} {mode} f{k}: peak core6 mIoU {h.val_miou_core6.max():.4f} @ e{em} "
                  f"of {last} | final gate {h.gate.iloc[-1]:.3f}{flag}")

## 22. Standalone inference from Hugging Face

Self-contained: paste into an empty notebook with nothing above it. The architecture is rebuilt from
the config embedded in `best.pt`, so the checkpoint alone is sufficient — which closes the ledger
item recording that `sugam24/geonusaf-s4-tcsegformer-R0-block-fold1` ships no `.py` and therefore
cannot be instantiated by a third party. **Push this cell's code to the repo as `modeling_tc.py`
when you release the dataset**, or the same problem recurs.

Preprocessing matches training exactly: `INTER_AREA` downsample to 512, BGR to RGB, `/255`, ImageNet
mean/std. Under the default `c6` scheme the head is 6-way and the model can never predict
"unlabelled" — every pixel gets one of the six land-use classes, including inside annotation gaps.
That is correct behaviour, not a bug, but it means a rendered prediction has no black regions while
the ground truth does; §23 masks them for display.

`transformers` is pinned below 5 because 5.x renames `decode_head.linear_c` to `linear_projections`,
which changes every checkpoint key.

In [ ]:
# ===================== STANDALONE -- paste into an empty notebook =====================
# !pip install -q "transformers<5" huggingface_hub
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, cv2
from huggingface_hub import hf_hub_download
from transformers import SegformerConfig, SegformerForSemanticSegmentation

REPO = "sugam24/geonusaf-tcsegformer-legacy-c6-tc_full-legacy_kfold-fold0"   # <-- any run
DEV = "cuda" if torch.cuda.is_available() else "cpu"

class _Detail(nn.Module):
    def __init__(s, chs):
        super().__init__(); c1, c2, c3 = chs
        b = lambda i, o, st: nn.Sequential(nn.Conv2d(i, o, 3, st, 1, bias=False),
                                           nn.GroupNorm(min(8, o), o), nn.ReLU(inplace=True))
        s.net = nn.Sequential(b(3, c1, 2), b(c1, c2, 1), b(c2, c3, 1)); s.out_ch = c3
    def forward(s, x): return s.net(x)

class _TC(nn.Module):
    def __init__(s, C):
        super().__init__()
        conf = SegformerConfig.from_pretrained(C["HF_MODEL_ID"]); conf.num_labels = C["N_CLASSES"]
        base = SegformerForSemanticSegmentation.from_pretrained(
            C["HF_MODEL_ID"], config=conf, ignore_mismatched_sizes=True)
        s.encoder, s.head = base.segformer, base.decode_head
        D = conf.decoder_hidden_size; s.use_detail = C["USE_DETAIL"]
        if s.use_detail:
            s.proj = nn.Conv2d(D, C["FUSE_CH"], 1)
            s.detail = _Detail(tuple(C["DETAIL_CH"]))
            s.gate = nn.Parameter(torch.tensor(float(C["DETAIL_GATE_INIT"])))
            s.fuse = nn.Sequential(
                nn.Conv2d(C["FUSE_CH"] + s.detail.out_ch, C["FUSE_CH"], 3, 1, 1, bias=False),
                nn.GroupNorm(8, C["FUSE_CH"]), nn.ReLU(inplace=True))
            s.refine = nn.Conv2d(C["FUSE_CH"], C["N_CLASSES"], 1)
    def forward(s, x):
        H, W = x.shape[-2:]
        hs = s.encoder(x, output_hidden_states=True, return_dict=True).hidden_states
        outs = []
        for h_, mlp in zip(hs, s.head.linear_c):
            hh, ww = h_.shape[2], h_.shape[3]
            t = mlp(h_).permute(0, 2, 1).reshape(h_.shape[0], -1, hh, ww)
            outs.append(F.interpolate(t, size=hs[0].shape[2:], mode="bilinear", align_corners=False))
        fused = s.head.dropout(s.head.activation(s.head.batch_norm(
            s.head.linear_fuse(torch.cat(outs[::-1], 1)))))
        logits = F.interpolate(s.head.classifier(fused), size=(H, W),
                               mode="bilinear", align_corners=False)
        if not s.use_detail:
            return logits
        f = F.interpolate(s.proj(fused), size=(H // 2, W // 2), mode="bilinear", align_corners=False)
        f = torch.cat([f, s.gate * s.detail(x)], 1)
        return logits + F.interpolate(s.refine(s.fuse(f)), size=(H, W),
                                      mode="bilinear", align_corners=False)

def _state(ck):
    for k in ("model_state", "model", "state_dict"):
        v = ck.get(k)
        if isinstance(v, dict) and v and all(torch.is_tensor(t) for t in v.values()): return v
    for v in ck.values():
        if isinstance(v, dict) and v and all(torch.is_tensor(t) for t in v.values()): return v
    raise KeyError("no tensor state dict in checkpoint")

ck = torch.load(hf_hub_download(REPO, "best.pt"), map_location="cpu", weights_only=False)
C = ck["cfg"]
net = _TC(C); net.load_state_dict(_state(ck), strict=True); net.to(DEV).eval()
print("loaded", REPO, "| recipe", ck.get("recipe"), "|", C["N_CLASSES"], "classes | core6 mIoU",
      round(ck["metrics"]["mIoU_core6"], 4), "| epoch", ck["epoch"])

MEAN = np.array([0.485, 0.456, 0.406], np.float32); STD = np.array([0.229, 0.224, 0.225], np.float32)
# index -> RGB, six land-use classes (a c6 model never predicts "unlabelled")
PAL = np.array([[128,0,0],[0,128,0],[0,0,128],[0,128,128],[128,128,0],[128,0,128],[0,0,0]], np.uint8)

def predict(path, size=512):
    im = cv2.imread(str(path), cv2.IMREAD_COLOR)
    im = cv2.resize(im, (size, size), interpolation=cv2.INTER_AREA)
    x = ((im[:, :, ::-1].astype(np.float32) / 255.0) - MEAN) / STD
    x = torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1)))[None].to(DEV)
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=(DEV == "cuda")):
        p = net(x).float().argmax(1)[0].cpu().numpy().astype(np.uint8)
    return p, PAL[p]
# =====================================================================================

## 23. Qualitative triptychs — image / GT / prediction

Rendered through the **clean** loader, so what you see is the deterministic prediction rather than a
randomly flipped one, and ranked by the per-image 6-class mIoU already computed in §19. The failure
cases are what reviewers want to see, and on this dataset they are diagnostic: if the worst tiles are
dominated by River/UnusedLand confusion, that is a spectral problem — dry sand and gravel bars are
RGB-identical to water and there is no NIR band — and no architectural change will fix it.

Under `IGNORE_UNLABELLED = True` the prediction panel is masked black wherever ground truth is
unlabelled. The model does emit a class there — it has no "unlabelled" output — but those pixels were
never trained on and are not scored, so showing them would be showing something the metric ignores.

The specific thing to check against the B0 figures: whether River and Road come out as coherent
ribbons or scattered pixels. The detail path exists to make thin structures survive the H/4
bottleneck, so if TC's roads are visibly more connected at equal IoU, that belongs in the paper as a
clDice row, not as an unsupported claim about "sharper" output.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

PALETTE_RGB = np.zeros((256, 3), np.uint8)
for bgr, idx in BGR_TO_CLASS.items():
    PALETTE_RGB[idx] = bgr[::-1]
PALETTE_RGB[VOID_LABEL] = (0, 0, 0)

def colorize(lab): return PALETTE_RGB[lab]

ranked = sorted(per_image, key=lambda d: d["mIoU_core6"])
picks_worst, picks_best = ranked[:3], ranked[-3:][::-1]

def triptych(items, title):
    fig, axes = plt.subplots(len(items), 3, figsize=(13, 4.3 * len(items)))
    axes = np.atleast_2d(axes)
    for r, rec in enumerate(items):
        i = rec["idx"]
        _, _, iname, mname = clean_ds.recs[i]
        img = cv2.cvtColor(cv2.imread(str(CACHE / "images" / iname)), cv2.COLOR_BGR2RGB)
        x, y, _ = clean_ds[i]
        gt = y.numpy().astype(np.uint8)
        with torch.no_grad(), torch.cuda.amp.autocast(enabled=cfg.AMP):
            pr = forward_logits(best_model, x[None].to(DEVICE)).float().argmax(1)[0]
        pr = pr.cpu().numpy().astype(np.uint8)
        if cfg.IGNORE_UNLABELLED:
            pr = np.where(gt == cfg.IGNORE_INDEX, cfg.IGNORE_INDEX, pr).astype(np.uint8)
        panels = [(img, iname), (colorize(gt), "ground truth"),
                  (colorize(pr), f"prediction (core6 mIoU {rec['mIoU_core6']:.3f})")]
        for c, (im, t) in enumerate(panels):
            axes[r, c].imshow(im); axes[r, c].set_title(t, fontsize=9); axes[r, c].axis("off")
    handles = [Patch(color=PALETTE_RGB[i] / 255, label=n) for i, n in enumerate(cfg.CLASS_NAMES)]
    handles.append(Patch(color="black", label="unlabelled"
                         + (" (ignored)" if cfg.IGNORE_UNLABELLED else "")))
    fig.legend(handles=handles, loc="lower center", ncol=7, fontsize=8, frameon=False)
    fig.suptitle(f"{title} - TC-SegFormer legacy ({cfg.RECIPE}, {SCHEME_TAG}), "
                 f"{cfg.SPLIT_MODE} fold {cfg.FOLD} (weights: {cfg.CLASS_WEIGHT_MODE})", fontsize=12)
    plt.tight_layout(rect=[0, 0.04, 1, 0.97])
    p = f"{cfg.OUT_DIR}/qual_{title.replace(' ','_')}_{tag}.png"
    plt.savefig(p, dpi=130, bbox_inches="tight"); plt.show()
    hf_push(p, os.path.basename(p), "qualitative")

triptych(picks_best, "best cases")
triptych(picks_worst, "worst cases")

## 24. Confusion matrix figure

Row-normalised, so the diagonal is producer's accuracy. Ignored pixels never entered the matrix, so
every row sums over scored pixels only.

The two cells that carry the paper's argument are **UnusedLand <-> Agricultural** and
**River -> UnusedLand**. Those are the known spectral confusions, and their magnitude is what the
discussion should be about.

In [ ]:
cmn = np.array(final["confusion_matrix_norm"])
N = cfg.N_CLASSES
fig, ax = plt.subplots(figsize=(7.6, 6.2))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(N)); ax.set_yticks(range(N))
ax.set_xticklabels(cfg.CLASS_NAMES, rotation=45, ha="right"); ax.set_yticklabels(cfg.CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("Ground truth")
ax.set_title(f"Row-normalized confusion - TC-SegFormer legacy ({cfg.RECIPE}, {SCHEME_TAG}), "
             f"{cfg.SPLIT_MODE} fold {cfg.FOLD}")
for i in range(N):
    for j in range(N):
        ax.text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if cmn[i, j] > 0.5 else "black")
fig.colorbar(im, fraction=0.046)
plt.tight_layout()
p = f"{cfg.OUT_DIR}/confusion_{tag}.png"
plt.savefig(p, dpi=150, bbox_inches="tight"); plt.show()
hf_push(p, os.path.basename(p), "confusion matrix")

## 25. Run checklist, deviations, and the experiment this is for

### Checklist
1. Internet **ON**; Secrets `HF_TOKEN` (write) + `WANDB_API_KEY`; set `cfg.HF_USER`.
2. §5 must print **channel-order check PASSED**.
3. §6 must print the CSA band as **46.90 px**. Any other number means `TILE_METRES` or `CSA_DMAX_M`
   was edited and this run is no longer comparable with the earlier TC runs.
4. §8 prints the variant-leakage percentage. Write it down. Under `legacy_kfold` it will be ~96%.
5. §10 must print a CSA weight range inside [0.25, 1.0] under `tc_*`, and exactly 1.0/1.0 under
   `legacy_pure`.
6. §11 must print **CE parity check PASSED**.
7. §12 must print `logits after upsample: (1, 6, 512, 512)` — or `(1, 7, ...)` with
   `IGNORE_UNLABELLED = False`. A 128 in the spatial dims invalidates everything below it.
8. First 30 epochs: `gate` must move away from 0.100. If it has not, stop — the detail path is inert
   and you are training B0 at TC's cost.
9. One fold per session, resume is automatic from `last.pt`. After all folds, run §21.

### Where this is not a literal transcription of the legacy recipe
Three are unavoidable, three are deliberate scoring choices inherited from the B0 rebuild, and one is
forced by the architecture.

| Deviation | Why | Effect on the number |
|---|---|---|
| `LR_MODE = "new_boosted"`: new modules at 3e-4, everything pretrained at 3e-5 | `refine` is initialised at std 1e-4 by design; at a flat 3e-5 it barely leaves init and TC degenerates into B0 | real and intended — set `"legacy_flat"` for the strict version |
| loss computed in fp32 outside `autocast` | Lovasz sorts per-pixel errors and NaNs in fp16 | none for CE; it is already promoted to fp32 by autocast |
| CE is a weighted mean over the CSA map, not a plain mean | this is TC component D | none under `legacy_pure`, where every weight is 1.0 |
| `INTER_AREA` instead of bilinear for the 2345 -> 512 downsample | reuses the shared cache; bilinear aliases thin roads at 4.6x | small, in this notebook's **favour** |
| seeded RNG | the legacy script seeded nothing and was not reproducible | none in expectation |
| `use_deterministic_algorithms(False)` | ~6x wall-clock otherwise; 300 epochs would not fit | none in expectation, and it makes FPS comparable with the B0 run |
| `IGNORE_UNLABELLED = True` | annotation mistakes are not a class; matches the benchmark and the TC notebook | should help the 6 real classes slightly |
| `VAL_USES_TRAIN_TF = False` | removes an augmentation lottery from checkpoint selection | small, reduces variance |
| `SELECTION_METRIC = "val_miou_core6"` | select on the metric you report, not a noisier proxy | small, reduces variance |
| `CSA_DMAX_M` stated as 14.07 m at a 703.5 m tile | the original 6.0 m at a 300 m tile was the superseded scale assumption | **zero** — both give a 46.90 px band |

Everything else — split, weights, loss family, optimizer, base LR, schedule, clipping, EMA, epochs,
patience, batch, workers, augmentation, normalisation — is literal.

### Known residual weaknesses, inherited from the legacy recipe
Not bugs; the recipe under test, named so the comparison is read correctly. No clipping with uncapped
inverse-frequency weights leaves the occasional very large step (watch `grad_norm_true`, and under
`tc_full` the Lovasz term rides on top of it). `drop_last=False` means the final train batch holds 3
images, a noisy gradient. Class weights are counted over train+val together, a mild prior leak. There
is no EMA, typically worth +0.5-1.5 mIoU on a model this size. The balanced sampler draws with
replacement, so an "epoch" no longer visits every tile once. Changing any of them turns a recipe
comparison into something else, so change them only after run C has reported.

### The experiment

Four runs, one field changed between them. Together they decompose the whole gap into named parts
rather than leaving it as "the new recipe is better".

| Run | Change | What it isolates |
|---|---|---|
| **A** | nothing (`tc_full`, `legacy_kfold`, 5 folds) | TC under the exact conditions that produced the legacy figure |
| **B** | `RECIPE = "legacy_pure"` | whether TC's loss components still earn their place under this recipe |
| **C** | `SPLIT_MODE = "block"`, `N_FOLDS = 3` | the reported protocol, legacy recipe — the publishable row |
| **D** | `SPLIT_MODE = "random"`, `N_FOLDS = 3` | separates variant leakage from spatial geography |

Reading it, with the B0 runs as the other half of each pair (§21 does the arithmetic):

- **A's delta vs B0-A** — does TC beat B0 under the legacy recipe? Valid as a paired comparison.
- **A minus B** — the contribution of CSA + sampler + Lovasz + clDice, holding architecture fixed.
- **C's delta vs B0-C** — the same question under the reported protocol. **This is the row for the
  paper**, and the one to compare against the existing block-fold figures (B0 0.3962, TC 0.4682).
- **A's delta minus C's delta** — if TC's advantage shrinks substantially when leakage is removed,
  part of what the detail path buys is memorisation of mirrored tiles, and that must be said.

### One thing not to do

Do not put a `legacy_kfold` mIoU in the paper, in either column, even with a footnote. A reviewer who
opens the training script will find `KFold` over 1224 files in about ninety seconds, and at that
point the question stops being about the architecture. The honest and more interesting version is the
one you already have data for: a benchmark where the split is grouped and blocked, with a stated
leakage gap, and "here is what the same model scores when the split is done naively" as a one-line
methodological aside. Framed as a caution it is a strength; framed as a result it is not survivable.

### If the number comes out disappointing

The most likely outcome, on the evidence already in hand, is that TC under the legacy recipe lands
near TC under the current recipe once scored the same way, and that most of the apparent B0
improvement was leakage plus the 7th class plus the batch estimator. That is not a wasted run — it is
the measurement that closes the question, and a decomposition table is a better thing to put in front
of a reviewer than an unexplained discrepancy between two of your own numbers.

The one genuinely new variable here is the frozen-augmentation fix. The previous TC runs trained on
effectively static augmentation for their whole length, and this one does not. If anything in this
notebook produces a real improvement over the existing block-fold TC figures, that is the first place
to look, and it is worth a one-line check: `RECIPE = "tc_full"`, `SPLIT_MODE = "block"`,
`MAX_EPOCHS = 120`, which is the current protocol's horizon with live augmentation and nothing else
changed.